# This document contains a few robot Controller **classes** 
* Each Robot Controller implements controller that can be executed at each time step $\Delta T_{ctrl}$, and recieved tha same `ObservableStates`, `ObservableDesiredStates` as the same class vector as an array of the computing horizon. 
* Each Robot controller contains `compute`, and `_init_` Functions that uses/passes the same variables back for drop in place substitution.  
* All numerical `simulation_paramaters` are not defined inside classes and are passes during `_init_`. All controllers use the same knowledge of the system.
* All robot classes pass 4 input commands (1 per wheel) and since controllers may work under diffrent step size, commands are held over steps $N = \Delta T_{model}/\Delta T_{ctrl}$ 
* All Controllers are designed as path following
* All convex ompimization follows DDL compliances to minimize python overhead setup time by using `cp.paramaters` and `cp.sum_squares` with pre-computed matrix square roots guaranteeing zero compilation overhead and warm-started OSQP convergence. 

1. `PIDCascadeController`:
    * Contains 5 internal `PIDClass' loops, one for the global motion and 4 PI controllers on individual wheels
    * Uses rigid chassis motion to prject the kimematics using instantanious `compute_Hmatrix` based on paramaters.
    * Protects the motion using maximum velocities 
    * Conditional anti-windup on integral state
    * Clamping on the output commands to the wheels 
    * Clamping on the acceleration of the robot in transmational and rotation commands to assist in slipping avoidance which can be disabled

2. `MPCControllerSimpleModel`:
    * Contains a simple linear idealized model of the plant where friction is infinate and ignored
    * All the states of the simple model are observable ($i , V, x, \omega$)
    * ignores $\omega$ states since in a simple model velocities $v$ and $\omega$ are related directly but in our simulation there is disturbances which are friction




In [14]:
class OmniRobotObservableState:
    """
    Data container of data that the controller observes at every time step.
    """
    def __init__(self):
        # Memory Integration States (Differential Cores)
        self.I = np.zeros(4)            # Armature currents [i1, i2, i3, i4] (A)
        self.omega = np.zeros(4)        # Wheel angular velocities [w1, w2, w3, w4] (rad/s)
        self.pos_global = np.zeros(3)   # Global coordinates [x, y, theta] (m, rad)
        self.V_actual = np.zeros(4)     # Clipped applied terminal voltages [V1, V2, V3, V4] (V)
        


## Closed-Loop Cascaded PID Controller Execution & Plant Stepping

This section explains the **Hierarchical Cascaded Closed-Loop PID Controller** and its control on the physical robot plant.

---

### Controller Architecture
* **multirate discrete-time architecture**: The cascaded PID runs every $\Delta T_{\text{ctrl}}$ and the Plant physics engine integrates electrical dynamics and LuGre friction every $\Delta T_{\text{plant}}$. Zero-Order Hold (ZOH)is computed for each controller update, the calculated motor commands $\mathbf{V}_{\text{cmd}} = [V_1, V_2, V_3, V_4]^\top$ are held constant across N consecutive plant integration substeps.
* **Serapare Position and Wheel PID Loops**: The Controller runs 5 seperate closed loop controllers. The first is on the position errors of $[X_g, Y_g, \Theta_g]$ which computes the body velocity commands.  These are then projected to the robot coordinate velocites and thus to the wheel velocites under "ideal" friction assumptions. 

```text
  Path Generator: Target Pose X_d(t) at 50 Hz
                     │
                     ▼
  ┌──────────────────────────────────────────────────────────────┐
  │ 1. OUTER POSITION PID & HEADING REGULATION (50 Hz)           │
  │    • Distance error: rho = ||X_d - X_actual||                │
  │    • Output: Desired linear speed v_linear                   │
  │    • Heading error: e_theta = (theta_d - theta)_wrapped      │
  │    • Rotates global vector to local body axes (+x_r, +y_r)   │
  └──────────────────────────────┬───────────────────────────────┘
                                 │
                                 ▼
  ┌──────────────────────────────────────────────────────────────┐
  │ 2. TRACTION PRESERVATION: TWO-AXIS SLEW LIMITER              │
  │    • Linear Acceleration Clamp: ||dv_trans|| <= a_max * dt   │
  │    • Angular Acceleration Clamp: |d_omega|  <= alpha_max * dt│
  │    • Protects contact patch bristles against slip breakaway  │
  └──────────────────────────────┬───────────────────────────────┘
                                 │
                                 ▼
  ┌──────────────────────────────────────────────────────────────┐
  │ 3. ASYMMETRIC KINEMATICS PROJECTION (H MATRIX)               │
  │    • omega_target = (1 / r) * H @ v_body_cmd                 │
  │    • Projects 3-DOF body motion to 4 wheel speed setpoints   │
  └──────────────────────────────┬───────────────────────────────┘
                                 │
                                 ▼ Target Wheel Speeds [omega_d1..4]
  ┌──────────────────────────────────────────────────────────────┐
  │ 4. INNER WHEEL SPEED PI REGULATORS (4 Controllers)           │
  │    • Error: e_omega_i = omega_d,i - omega_actual,i           │
  │    • Conditional Anti-Windup Clamping on integral state      │
  │    • Output: Commanded Terminal Voltages V_cmd = [V1..V4]    │
  └──────────────────────────────┬───────────────────────────────┘
                                 │
                                 ▼ Zero-Order Hold (Held for 20 plant steps)
  ┌──────────────────────────────────────────────────────────────┐
  │ 5. PHYSICAL PLANT ENGINE (20x 1 kHz Substeps = 20 ms)        │
  │    • Motor Armature Current Dynamics: di/dt = (V - Ri - e)/Lm│
  │    • Dynamic Weight Transfer: Normal force pitch/roll shifts │
  │    • LuGre Bristle Friction: Evaluates presliding & slip v_s │
  │    • Newton-Euler Kinetics: Advances robot pose X(t + 20ms)  │
  └──────────────────────────────────────────────────────────────┘
```




### `PIDController` Class Architecture

The `PIDController` class provides an industrial-grade, discrete-time Proportional-Integral-Derivative regulator. It incorporates **dynamic sample-time handling**, **output saturation limiting**, and **conditional anti-windup clamping**.

#### Mathematical Formulation Implemented:
* **Error**: $e_k = r_k - y_k$
* **Proportional**: $P_k = K_p \cdot e_k$
* **Integrator Candidate**: $I_{\text{cand}, k} = I_{k-1} + e_k \cdot \Delta t$
* **Derivative**: $D_k = K_d \cdot \frac{e_k - e_{k-1}}{\Delta t}$
* **Control Effort**: $u_{raw-k} = P_k + Ki * I_{cand,k} + D_k$ and $u_k = clip(u_{raw,k}, u_{min}, u_{max})  $
* **Anti-Windup Clamping**: Freezes integration when output saturates in the direction of the error ($u_{\text{raw}} \ne u$ and $\text{sign}(e_k) == \text{sign}(u_{\text{raw}})$).

#### Class Methods:
* **`__init__(kp, ki, kd, dt, output_limits, integral_limits)`**: Configures feedback gains, loop time $\Delta t$, and saturation bounds.
* **`reset()`**: Clears accumulator states ($I = 0$, $e_{k-1} = 0$, $P = 0, D = 0$).
* **`compute(error, dt)`**: Executes the discrete update cycle and returns saturated command $u_k$.


In [15]:
import numpy as np

class PIDController:
    """
    Discrete-time PID controller with integral anti-windup clamping and output saturation.
    
    Mathematical Formulation:
      e[k] = r[k] - y[k]                                  (Tracking error)
      P[k] = Kp * e[k]                                    (Proportional action)
      I_cand[k] = I[k-1] + e[k] * dt                      (Unclamped integral candidate)
      D[k] = Kd * (e[k] - e[k-1]) / dt                    (Derivative action via backward difference)
      u_raw[k] = P[k] + Ki * I_cand[k] + D[k]             (Unsaturated control effort)
      u[k] = clip(u_raw[k], u_min, u_max)                 (Actuator saturation clipping)
      
    Anti-Windup Clamping Logic:
      If u_raw[k] != u[k] and sign(e[k]) == sign(u_raw[k]):
          I[k] = I[k-1]                                   (Freeze integration when saturated in error direction)
      Else:
          I[k] = clip(I_cand[k], I_min, I_max)            (Permit normal accumulation or recovery)
    """
    def __init__(self, kp: float, ki: float, kd: float, dt: float = 0.02, 
                 output_limits: tuple = None, integral_limits: tuple = None):
        """
        Initializes the PID controller parameters, memory states, and saturation boundaries.
        
        Parameters:
            kp (float): Proportional gain constant (Kp).
            ki (float): Integral gain constant (Ki).
            kd (float): Derivative gain constant (Kd).
            dt (float): Nominal discrete loop execution period (default: 0.02s = 50 Hz).
            output_limits (tuple, optional): (u_min, u_max) bounds on final actuator output.
            integral_limits (tuple, optional): (I_min, I_max) bounds on the integral state accumulator.
        """
        self.kp = float(kp)
        self.ki = float(ki)
        self.kd = float(kd)
        self.dt = float(dt)
        
        # Actuator saturation limits [u_min, u_max]
        self.output_limits = output_limits
        
        # Dedicated integrator accumulator limits [I_min, I_max]
        self.integral_limits = integral_limits
        
        # Internal persistent state memory
        self.integral = 0.0        # Accumulated integral sum I[k]
        self.previous_error = 0.0  # Historical error e[k-1] for derivative estimation
        
        # Diagnostics / Telemetry terms logged per step
        self.p_term = 0.0          # P[k] = Kp * e[k]
        self.i_term = 0.0          # I_term[k] = Ki * I[k]
        self.d_term = 0.0          # D[k] = Kd * de/dt
        self.last_output = 0.0     # Final saturated output u[k]

    def reset(self):
        """
        Resets the internal integrator memory, historical error, and diagnostic terms.
        Must be invoked at the start of new trajectories to avoid historical state contamination.
        """
        self.integral = 0.0
        self.previous_error = 0.0
        self.p_term = 0.0
        self.i_term = 0.0
        self.d_term = 0.0
        self.last_output = 0.0

    def compute(self, error: float, dt: float = None) -> float:
        """
        Executes one discrete PID update step given the tracking error e[k].
        
        Parameters:
            error (float): Current system tracking error e[k] = target - actual.
            dt (float, optional): Time interval since the previous compute call (defaults to self.dt).
            
        Returns:
            output (float): Saturated, anti-windup clamped control signal u[k].
        """
        # Enforce valid positive time delta to prevent division by zero
        if dt is None:
            dt = self.dt
        if dt <= 0.0:
            dt = 1e-4

        # 1. Proportional Term calculation: P[k] = Kp * e[k]
        self.p_term = self.kp * error

        # 2. Derivative Term calculation (Backward difference): D[k] = Kd * (e[k] - e[k-1]) / dt
        derivative = (error - self.previous_error) / dt
        self.d_term = self.kd * derivative

        # 3. Unclamped Integral Candidate: I_cand[k] = I[k-1] + e[k] * dt
        integral_candidate = self.integral + error * dt

        # 4. Compute unsaturated raw control effort candidate: u_raw[k] = P + Ki * I_cand + D
        candidate_output = self.p_term + self.ki * integral_candidate + self.d_term

        # 5. Evaluate Actuator Output Saturation Clamping: u[k] = clip(u_raw, u_min, u_max)
        if self.output_limits is not None:
            output_min, output_max = self.output_limits
            output = np.clip(candidate_output, output_min, output_max)
        else:
            output = candidate_output

        # 6. Evaluate Conditional Anti-Windup Clamping:
        # If output is saturated AND the error is driving the system deeper into saturation,
        # freeze the integrator (hold I[k] = I[k-1]). Otherwise, accept the update.
        if self.output_limits is not None:
            is_saturated = (candidate_output != output)
            same_sign = (np.sign(error) == np.sign(candidate_output))
            if is_saturated and same_sign:
                # Anti-windup active: freeze integration
                pass
            else:
                # Normal integration: update integrator accumulator
                self.integral = integral_candidate
        else:
            self.integral = integral_candidate

        # 7. Apply optional independent integrator bounds [I_min, I_max]
        if self.integral_limits is not None:
            i_min, i_max = self.integral_limits
            self.integral = float(np.clip(self.integral, i_min, i_max))

        # Store diagnostic telemetry and state memory
        self.i_term = self.ki * self.integral
        self.previous_error = error
        self.last_output = output

        return float(output)


###  `CascadedPIDController` Class Architecture

The `CascadedPIDController` class encapsulates the full hierarchical closed-loop architecture for the 4-omniwheel vehicle operating at $\Delta T_{ctrl}$.


* **Hierarchical Cascade (6 PID Regulators)**:
  1. **1 Outer Position PID (`self.pid_position`)**: Regulates scalar Euclidean distance error $\rho = \sqrt{\Delta x^2 + \Delta y^2}$ to command desired linear speed magnitude $v_{\text{linear}}$ directed along the target line-of-sight.
  2. **4 Inner Wheel Speed PIDs (`self.pid_wheels`)**: Independent PI velocity regulators driving wheels 1 through 4 via terminal voltages $\mathbf{V}_{\text{cmd}} = [V_1, V_2, V_3, V_4]^\top$.
  3. **1 Proportional Orientation controller**: Regulates orientation. 

* **Coordinate Transformations**:
  * Heading error $e_\theta = (\theta_d - \theta)$ is wrapped to $[-\pi, \pi]$ to ensure shortest-arc angular turning.
  * Commanded global velocity $[v_{x,g}, v_{y,g}]$ is rotated into the local vehicle body frame ($+x_r$ Right, $+y_r$ Forward) via $R^\top(\theta)$.

* **Software Limiting**:
  * **Linear Acceleration ($a_{\max}$)**: Vector-norm rate limiter preserves trajectory heading while bounding linear acceleration below $\mu_s g$.
  * **Angular Acceleration ($\alpha_{\max}$)**: Bounded slew limiter ($\alpha_{\max}$) prevents sudden rotational shocks from destabilizing tractive adhesion during compound maneuvers.
  * **Contraints on Voltage**: clips input at the maximum allawable voltage, the plant must inherantly handle the current constaints

* **Asymmetric Kinematics Mapping ($\mathbf{H}$)**:
  * Projects 3-DOF body twist into individual wheel speed targets: $\boldsymbol{\omega}_{\text{target}} = \frac{1}{r} \mathbf{H} \, \mathbf{v}_{\text{body, cmd}}$.

#### Class Methods:
* **`__init__(simulation_params, pos_pid_gains, wheel_pid_gains, dt, v_max, a_max, alpha_max, ...)`**: Instantiates all 5 PID controllers, builds kinematics matrix $\mathbf{H}$, and configures limits.
* **`reset()`**: Resets all 5 internal PID controllers and clears historical velocity slew memory.
* **`compute(target_states, current_states, dt)`**: Executes the full cascaded calculation cycle.


In [ ]:
import numpy as np

class CascadedPIDController:
    """
    Cascaded Closed-Loop Control Architecture for the Asymmetric 4-Omniwheel Robot.
    
    Structure:
      Level 1: Outer-Loop Position PID (Distance rho -> Commanded body velocity)
      Level 2: Software Slew-Rate Limiter (Enforces a_max and alpha_max traction protection)
      Level 3: Asymmetric Kinematics Projection (H matrix: Body twist -> Wheel speed targets)
      Level 4: Inner-Loop 4x Wheel Speed PI Regulators (Wheel speed error -> Terminal voltages)
    """
    def __init__(self, simulation_params: dict, 
                 pos_pid_gains: dict = None, 
                 wheel_pid_gains: dict = None, 
                 dt: float = 0.02,
                 a_max: float = 2.0,
                 alpha_max: float = 5.0,
                 k_theta: float = 2.0,
                 voltage_limit: float = 12.0):
        """
        Parameters:
            simulation_params (dict): Physical vehicle parameters (r, Rb, alpha_f, alpha_b).
            pos_pid_gains (dict): Gains {'kp', 'ki', 'kd'} for outer position PID controller.
            wheel_pid_gains (dict): Gains {'kp', 'ki', 'kd'} for inner 4 wheel speed PIDs.
            dt (float): Discrete controller execution period (default: 0.02s = 50 Hz).
            a_max (float): Software maximum linear acceleration slew limit (m/s^2).
            alpha_max (float): Software maximum angular acceleration slew limit (rad/s^2).
            k_theta (float): Heading error proportional gain (1/s).
            voltage_limit (float): Maximum allowable motor terminal voltage command (Volts).
        """
        self.p = simulation_params.copy()  # Preserve original simulation parameters
        self.dt = float(dt)
        self.a_max = float(a_max)
        self.alpha_max = float(alpha_max)
        self.k_theta = float(k_theta)
        self.voltage_limit = float(voltage_limit)
        
        # Extract vehicle geometry for kinematic mapping
        self.r = self.p['r']
        self.Rb = self.p['Rb']
        self.alpha_f = self.p['alpha_f']
        self.alpha_b = self.p['alpha_b']
        self.H = self._build_kinematics_matrix()
        
        # Default outer-loop position PID gains if not specified
        if pos_pid_gains is None:
            pos_pid_gains = {'kp': 1.5, 'ki': 0.1, 'kd': 0.05}
        if wheel_pid_gains is None:
            # Inner wheel loops operate as pure PI regulators (Kd=0 prevents encoder derivative kick)
            wheel_pid_gains = {'kp': 0.15, 'ki': 0.8, 'kd': 0.0}
            
        # 1. Instantiate Controller 1 of 5: Outer-Loop Position PID (Outputs linear speed in m/s)
        self.pid_position = PIDController(
            kp=pos_pid_gains['kp'],
            ki=pos_pid_gains['ki'],
            kd=pos_pid_gains['kd'],
            dt=self.dt
        )
        
        # 2. Instantiate Controllers 2..5 of 5: Inner-Loop Wheel Speed PI Regulators (Outputs terminal Volts)
        self.pid_wheels = [
            PIDController(
                kp=wheel_pid_gains['kp'],
                ki=wheel_pid_gains['ki'],
                kd=wheel_pid_gains['kd'],
                dt=self.dt,
                output_limits=(-self.voltage_limit, self.voltage_limit),  # Bounded by motor supply voltage
                integral_limits=(-self.voltage_limit, self.voltage_limit)
            ) for _ in range(4)
        ]
        
        # Memory tracking state for slew rate acceleration limiting [vx_past, vy_past, omega_yaw_past]
        self.v_body_cmd_past = np.zeros(3)

    def _build_kinematics_matrix(self) -> np.ndarray:
        """
        Constructs the exact 4x3 asymmetric kinematic mapping matrix H:
          v_wheel_rolling = H @ [vx_local, vy_local, omega_yaw]^T
        """
        af = self.alpha_f
        ab = self.alpha_b
        Rb = self.Rb
        return np.array([
            [-np.sin(af),  np.cos(af), Rb],  # Wheel 1: Front Right (Quadrant I)
            [-np.sin(af), -np.cos(af), Rb],  # Wheel 2: Front Left  (Quadrant II)
            [ np.sin(ab), -np.cos(ab), Rb],  # Wheel 3: Rear Left   (Quadrant III)
            [ np.sin(ab),  np.cos(ab), Rb]   # Wheel 4: Rear Right  (Quadrant IV)
        ])

    def reset(self):
        """
        Resets all 5 internal PID controllers and clears historical velocity slew memory.
        Ensures smooth initial step response when initiating a new trajectory.
        """
        self.pid_position.reset()
        for w_pid in self.pid_wheels:
            w_pid.reset()
        self.v_body_cmd_past = np.zeros(3)

    def compute(self, target_pose: np.ndarray, current_state: OmniRobotObservableState) -> np.ndarray:
        """
        Executes the full cascaded control loop step at 50 Hz.
        
        Parameters:
            target_pose (np.ndarray): Target world pose [x_d, y_d, theta_d] (meters, radians).
            current_state (OmniRobotObservableState): Current plant observale state from which we need position [x, y, theta] (meters, radians).
            dt (float, optional): Time interval since last execution (default: self.dt = 0.02s).
            
        Returns:
            V_cmd (np.ndarray): Commanded motor terminal voltages [V1, V2, V3, V4] (Volts).
        """
        
        ## Unpack target and current states for clarity
        x_d, y_d, theta_d = target_pose
        x, y, theta = current_state.pos_global
        current_omega = current_state.omega  # Current wheel angular velocities [w1, w2, w3, w4]
        
        # =========================================================================
        # LEVEL 1: TRACKING ERROR EVALUATION (WORLD FRAME)
        # =========================================================================
        # 1a. Spatial Euclidean position displacement: rho = sqrt(dx^2 + dy^2)
        dx = x_d - x
        dy = y_d - y
        rho = np.hypot(dx, dy)
        
        # 1b. Heading angle error wrapped to [-pi, pi] for shortest rotational arc:
        # e_theta = (theta_d - theta + pi) % (2*pi) - pi
        e_theta = (theta_d - theta + np.pi) % (2.0 * np.pi) - np.pi
        
        # =========================================================================
        # LEVEL 2: OUTER-LOOP POSITION PID CONTROLLER (CONTROLLER 1 OF 5)
        # =========================================================================
        # Compute commanded linear speed magnitude based on distance error rho
        v_linear_cmd = self.pid_position.compute(rho, self.dt)
        
        # Project linear speed along the line-of-sight unit vector in world coordinates
        if rho > 1e-4:
            vx_global_cmd = v_linear_cmd * (dx / rho)
            vy_global_cmd = v_linear_cmd * (dy / rho)
        else:
            vx_global_cmd = 0.0
            vy_global_cmd = 0.0
            
        # Rotate global velocity into robot local body frame (+x_r Right, +y_r Forward): v_local = R(theta)^T v_global
        # GLOBAL FRAME: with the plant's rotation
        v_local_cmd = OmniRobotSimulationODESolve._rotation(theta).T @ np.array([vx_global_cmd, vy_global_cmd, 0.0])
        vx_local_cmd = v_local_cmd[0]
        vy_local_cmd = v_local_cmd[1]
        
        # Compute raw proportional yaw rate and apply optional saturation limit (if desired)
        omega_yaw_cmd = self.k_theta * e_theta
        
        # Compute the full body command vector in local chassis frame: [vx_local, vy_local, omega_yaw]
        raw_v_body_cmd = np.array([vx_local_cmd, vy_local_cmd, omega_yaw_cmd])
        
        # =========================================================================
        # LEVEL 3: TWO-AXIS SOFTWARE SLEW-RATE LIMITING (TRACTION SLIP PROTECTION)
        # =========================================================================
        # 3a. Linear Acceleration Vector Slew Limiter (a_max in m/s^2):
        # Prevents dynamic weight transfer spikes (pitch/roll) from breaking tire grip
        max_dv_trans = self.a_max * self.dt
        dv_trans = raw_v_body_cmd[:2] - self.v_body_cmd_past[:2]
        dv_mag = np.hypot(dv_trans[0], dv_trans[1])
        if dv_mag > max_dv_trans and dv_mag > 0.0:
            # Scale vector uniformly to preserve exact trajectory motion direction
            dv_trans = dv_trans * (max_dv_trans / dv_mag)
        v_body_trans = self.v_body_cmd_past[:2] + dv_trans
        
        # 3b. Angular Acceleration Slew Limiter (alpha_max in rad/s^2):
        # Prevents high rotational jerk from breaking traction during compound maneuvers
        max_d_yaw = self.alpha_max * self.dt
        dv_yaw = np.clip(raw_v_body_cmd[2] - self.v_body_cmd_past[2], -max_d_yaw, max_d_yaw)
        v_body_yaw = self.v_body_cmd_past[2] + dv_yaw
        
        # Assemble rate-limited local chassis twist vector
        v_body_cmd = np.array([v_body_trans[0], v_body_trans[1], v_body_yaw])
        self.v_body_cmd_past = v_body_cmd
        
        # =========================================================================
        # LEVEL 4: ASYMMETRIC KINEMATIC PROJECTION (BODY TWIST -> WHEEL SPEEDS)
        # =========================================================================
        # omega_target = (1 / r) * H @ v_body_cmd
        omega_target = (1.0 / self.r) * (self.H @ v_body_cmd)
        
        # =========================================================================
        # LEVEL 5: INNER-LOOP 4x WHEEL SPEED PI REGULATORS (CONTROLLERS 2..5 OF 5)
        # =========================================================================
        # Each wheel PI evaluates speed error e_omega = omega_target[i] - current_omega[i]
        # and outputs motor terminal command V_cmd[i] bounded by motor supply limits
        V_cmd = np.zeros(4)
        for i in range(4):
            e_omega_i = omega_target[i] - current_omega[i]
            V_cmd[i] = self.pid_wheels[i].compute(e_omega_i, self.dt)
            
        return V_cmd


###  `MPCControllerSimple` Class Architecture & Mathematical Subfunction Design

This section establishes the detailed architectural blueprint and mathematical mechanics for the `MPCController` class. The controller is formulated as an online **Convex Quadratic Program (QP)** solved via **CVXPY** with the **OSQP** engine at a control rate ($\Delta T_{\text{ctrl}} $).

---

### Optimization Breakthroughs & Core Design Principles

1. **Integration with `OmniRobotNominalModel`:**  
   The controller completely delegates all continuous kinematics, electromechanical back-EMF coupling, and Zero-Order Hold (ZOH) discretization to the **`OmniRobotNominalModel`** class. The MPC never steps an internal numerical plant forward, it maintaining a well-conditioned linear state-space prediction model.

2. **Full Electromechanical 10-State Space:**  
   By tracking the 4 motor armature currents alongside global pose and global chassis velocities (changed in this copy: they were local, see the nominal model):
   $$\mathbf{x} = [x, \, y, \, \theta, \, \dot{x}_r, \, \dot{y}_r, \, \dot{\theta}, \, i_1, \, i_2, \, i_3, \, i_4]^\top \in \mathbb{R}^{10}$$
   The controller enforces **hard linear constraints on both motor terminal voltages and motor driver currents**, protecting actuators from thermal overload and driver trips during extreme maneuvers.

3. **Successive Linearization along Predicted Heading Trajectory (RTI):**  
   Since the model ignores friction, the kinematic motion is linear and $(\mathbf{A}, \mathbf{B})$ at each optimization cycle only depend on the orientation of the rotaion matrix.  The controller computes at each look-ahead step $j$ the matrixes around the **predicted chassis heading $\hat{\theta}_j$**. Step-specific discrete matrices $(\mathbf{A}_{d, j}, \mathbf{B}_{d, j})$ are computed via `nominal_model.get_discrete_matrices()`.

4. **Pre-Compiled Disciplined Parameterized Programming (DPP):**  
   The symbolic optimization problem (`cp.Problem`) is compiled **once** at startup. Decision variables ($\mathbf{X}, \mathbf{U}$) and parameters ($\mathbf{x}_0, \mathbf{A}_j, \mathbf{B}_j, \mathbf{X}_{\text{ref}}, \mathbf{V}_{\text{ref}}, \mathbf{u}_{\text{prev}}$) are allocated once. At each step, parameter arrays are updated in-place with zero memory allocation.
   Using matrix square roots ($\mathbf{Q}^{1/2}$) and the `cp.sum_squares` cost:
   $$\mathbf{e}^\top \mathbf{Q} \mathbf{e} = \|\mathbf{Q}^{1/2} \mathbf{e}\|_2^2 = \texttt{cp.sum\_squares}(\mathbf{Q}^{1/2} @ \mathbf{e})$$
   ensures strict DPP curvature verification across all CVXPY versions, guaranteeing instant, warm-started OSQP parameter updates without warnings or recompilation.

5. **Hardware Abstraction Layer (`OmniRobotObservableState`):**  
   The controller strictly interfaces with observable sensor signals (`pos_global`, `omega`, `I`, `V_actual`), maintaining a realistic hardware boundary that does not rely on hidden plant physics (bristles or normal load shifts).

6. **Dual Velocity Estimators:**  
   Because chassis velocity is not directly measured by a sensor, the controller integrates two complementary reconstruction models:
   * **`PoseVelocityEstimator` (Pose Differentiation):** Differentiates successive global coordinate readings $\mathbf{p}_k = [x, y, \theta]^\top$ and unwraps heading phase $[-\pi, \pi]$. The velocity stays in the global frame, the MPC's velocity state (changed in this copy: it was rotated into the body frame).
   * **`EncoderKinematicEstimator` (Wheel Odometry):** Inverts 4 wheel encoder speeds $\boldsymbol{\omega}$ via the Moore-Penrose kinematic pseudoinverse $\mathbf{H}^\dagger = (\mathbf{H}^\top \mathbf{H})^{-1} \mathbf{H}^\top$.
7. **Contact Slip & Disturbance Diagnostic:**  
   Comparing the two estimates yields the contact patch slip vector:
   $$\mathbf{v}_{\text{slip}} = \mathbf{H} \mathbf{v}_{\text{est, pose}} - r \boldsymbol{\omega}$$
   When $\|\mathbf{v}_{\text{slip}}\| \approx 0$, pure rolling traction holds. Deviations immediately identify wheel spinout, skidding, or unmodeled friction drag.

---

###  Subfunction Planning Overview
```text
┌────────────────────────────────────────────────────────────────────────────────────────┐
│                                   MPCController Class                                  │
├────────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                        │
│  1. `__init__(simulation_params, horizon_K=8, dt=0.02, ...)`                           │
│     • Instantiates `OmniRobotNominalModel(self.p)` (10 States, 4 Inputs).              │
│     • Instantiates `PoseVelocityEstimator` and `EncoderKinematicEstimator`.            │
│     • Pre-computes weighting matrix square roots (Q_pos_sqrt, Q_vel_sqrt, etc.).       │
│     • Calls `initializeOptimizationProblem()` and `reset()`.                           │
│                                                                                        │
│  2. `initializeOptimizationProblem()`                                                  │
│     • Allocates CVXPY decision variables: X_var (10, K+1), U_var (4, K).               │
│     • Allocates step-varying parameter matrices: A_params[K], B_params[K].             │
│     • Formulates velocity tracking error, voltage effort, and slew rate costs.         │
│     • Enforces ±12V voltage limits, ±5V/step slew limits, and ±20A current limits.     │
│     • Pre-compiles symbolic problem ONCE via cp.Problem(cp.Minimize(cost), constraints)│
│                                                                                        │
│  3. `reset()`                                                                          │
│     • Resets previous voltage memory (u_prev = 0), status flags, and estimators.      │
│                                                                                        │
│  4. `compute_optimal_u0(obs, pos_h, vel_h)` (Alias: `find_optimal_u0`)                 │
│     • Ingests OmniRobotObservableState (or legacy measured arrays).                    │
│     • Reconstructs global velocity via PoseVelocityEstimator (and local via the encoders).│
│     • Calculates contact patch slip diagnostic vector v_slip.                          │
│     • Updates predicted heading trajectory theta_pred[j] (Real-Time Iteration).        │
│     • Injects discrete matrices Ad_j, Bd_j into A_params[j], B_params[j] in-place.     │
│     • Solves pre-compiled QP in ~6-7 ms and returns first optimal voltage command u_0*.│
│                                                                                        │
└────────────────────────────────────────────────────────────────────────────────────────┘
```

### `OmniRobotNominalModel` Class Architecture & Internal Prediction Formulation

This section establishes the mathematical architecture and software implementation for the **`OmniRobotNominalModel`** class. Designed as an independent, lightweight control model, it provides the **internal linearizing prediction engine** for the Model Predictive Controller (MPC).

---

#### 1. **The Control-Oriented Solution:**  
The MPC **does not need to step forward through an internal physics plant**. Instead, the relationship between future inputs and states across the look-ahead horizon is governed directly by the discrete linear state-space equation:
$$\mathbf{x}_{j+1} = \mathbf{A}_d(\theta_k)\,\mathbf{x}_j + \mathbf{B}_d(\theta_k)\,\mathbf{u}_j$$
The `OmniRobotNominalModel` computes the exact Zero-Order Hold (ZOH) matrices $(\mathbf{A}_d, \mathbf{B}_d)$ **analytically via matrix exponential (`scipy.linalg.expm`)**, supplying them directly to the convex QP solver (OSQP).

---

### 2. State & Input Vector Parameterization

To explicitly enforce hard physical constraints on **both motor terminal voltages and armature winding currents**, the nominal state vector incorporates chassis kinematics, rigid-body kinetics, and electrical motor dynamics:

$$\mathbf{x} = \begin{bmatrix} \mathbf{p} \\ \mathbf{v} \\ \mathbf{i} \end{bmatrix} \in \mathbb{R}^{10}, \qquad \mathbf{u} = \begin{bmatrix} V_1 \\ V_2 \\ V_3 \\ V_4 \end{bmatrix} \in \mathbb{R}^4$$

* **Global Chassis Pose (3 States):** $\mathbf{p} = [x, \, y, \, \theta]^\top$ (meters, radians)
* **Global Chassis Velocities (3 States):** $\mathbf{v} = [\dot{x}_g, \, \dot{y}_g, \, \dot{\theta}]^\top$ (m/s, rad/s). **Changed in this copy: they were local.** With local velocities the model also needs the turning of the robot's axes, $\dot\theta\,[\dot y_r, -\dot x_r, 0]^\top$, which is not linear and was left out; a robot that moves while it turns was then predicted on the wrong path. With global velocities Newton's law needs no extra term; the heading only enters through $\mathbf{R}(\theta)$, which the MPC already evaluates at each predicted heading.
* **Motor Armature Currents (4 States):** $\mathbf{i} = [i_1, \, i_2, \, i_3, \, i_4]^\top$ (Amperes)
* **Actuator Control Inputs (4 Inputs):** $\mathbf{u} = [V_1, \, V_2, \, V_3, \, V_4]^\top$ (Volts)

---

### 3. Continuous-Time Equations Linearized with Respect to Heading ($\theta$)

The continuous electromechanical state space is formulated as:
$$\dot{\mathbf{x}}(t) = \mathbf{A}_c(\theta)\,\mathbf{x}(t) + \mathbf{B}_c\,\mathbf{u}(t)$$

$$\mathbf{A}_c(\theta) = \begin{bmatrix} 
\mathbf{0}_{3 \times 3} & \mathbf{I}_{3 \times 3} & \mathbf{0}_{3 \times 4} \\ 
\mathbf{0}_{3 \times 3} & \mathbf{0}_{3 \times 3} & \mathbf{R}(\theta)\,\mathbf{A}_{vi} \\ 
\mathbf{0}_{4 \times 3} & \mathbf{A}_{iv}\,\mathbf{R}(\theta)^\top & \mathbf{A}_{ii} 
\end{bmatrix}_{10 \times 10}, \qquad 
\mathbf{B}_c = \begin{bmatrix} 
\mathbf{0}_{3 \times 4} \\ 
\mathbf{0}_{3 \times 4} \\ 
\mathbf{B}_i 
\end{bmatrix}_{10 \times 4}$$

Where the constant physical blocks are defined by:
1. **Coordinate Rotation Matrix:** $\mathbf{R}(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta & 0 \\ \sin\theta & \cos\theta & 0 \\ 0 & 0 & 1 \end{bmatrix}$ (the plant's `_rotation`): local to global. $\mathbf{R}(\theta)\,\mathbf{A}_{vi}$ rotates the wheel forces into the global frame, $\mathbf{R}(\theta)^\top$ turns the global velocity into the local one for the back-EMF.
2. **Current-to-Acceleration Matrix:** $\mathbf{A}_{vi} = \frac{K_t}{r} \mathbf{M}^{-1} \mathbf{H}^\top \in \mathbb{R}^{3 \times 4}$
3. **Back-EMF Velocity Coupling Matrix:** $\mathbf{A}_{iv} = -\frac{K_b}{L_m r} \mathbf{H} \in \mathbb{R}^{4 \times 3}$
4. **Armature Resistance Decay:** $\mathbf{A}_{ii} = -\frac{R}{L_m} \mathbf{I}_{4 \times 4} \in \mathbb{R}^{4 \times 4}$
5. **Voltage-to-Current Scaling:** $\mathbf{B}_i = \frac{1}{L_m} \mathbf{I}_{4 \times 4} \in \mathbb{R}^{4 \times 4}$

---

### 4. Exact Discrete-Time Discretization (Van Loan Method)

The controller computes the exact discrete transition matrices $(\mathbf{A}_d(\theta_k), \mathbf{B}_d(\theta_k))$ across sampling period $\Delta t = 20\text{ ms}$ using the block matrix exponential:
$$\exp\left( \begin{bmatrix} \mathbf{A}_c(\theta_k) & \mathbf{B}_c \\ \mathbf{0}_{4 \times 10} & \mathbf{0}_{4 \times 4} \end{bmatrix} \Delta t \right) = \begin{bmatrix} \mathbf{A}_d(\theta_k) & \mathbf{B}_d(\theta_k) \\ \mathbf{0}_{4 \times 10} & \mathbf{I}_{4 \times 4} \end{bmatrix}$$

---

### 5. Linear Actuator Hardpoint Constraints Enforced in QP

Because voltages and currents are directly accessible in $\mathbf{u}$ and $\mathbf{x}$, the MPC enforces:
* **Terminal Voltage Ceiling:** $-V_{\max} \le \mathbf{u}_j \le V_{\max}$ ($\pm 12.0\text{ V}$)
* **Voltage Slew-Rate Bound:** $-\Delta V_{\max} \le \mathbf{u}_j - \mathbf{u}_{j-1} \le \Delta V_{\max}$ ($\pm 5.0\text{ V/step}$)
* **Motor Driver Over-Current Hardpoint:** $-I_{\max} \le \mathbf{x}_{6:10, \, j} \le I_{\max}$ ($\pm 20.0\text{ A}$)


In [33]:
class OmniRobotNominalModel:
    """
    10-State Control-Oriented Nominal Model for an Asymmetric 4-Omniwheel Robot.
    
    Acts as the algebraic linearizing prediction engine for Model Predictive Control (MPC).
    Eliminates the requirement for internal simulation plant forward stepping.
    
    State Vector (10):
      x[:3]   = [x, y, theta] (World coordinate pose in meters, rad)
      x[3:6]  = [vx_g, vy_g, omega_yaw] (GLOBAL chassis velocities in m/s, rad/s; changed in this copy, was local)
      x[6:10] = [i1, i2, i3, i4] (Motor armature currents in Amperes)
      
    Input Vector (4):
      u       = [V1, V2, V3, V4] (Motor terminal voltages in Volts)
      
    Subfunctions:
      1. __init__: Ingests parameters, builds kinematics H, and pre-computes constant 
                   coupling blocks (A_vi, A_iv, A_ii, B_c).
      2. get_rotation_matrix: Evaluates the 3x3 planar rotation matrix R(theta).
      3. get_continuous_matrices: Evaluates continuous state-space matrices Ac(theta) and Bc.
      4. get_discrete_matrices: Computes exact discrete matrices (Ad, Bd) via matrix exponential.
    """
    def __init__(self, simulation_params: dict):
        self.p = simulation_params.copy()
        
        # Extract vehicle physical constants
        self.Mr = float(self.p['Mr'])
        self.Jzz = float(self.p['Jzz'])
        self.Rb = float(self.p['Rb'])
        self.r = float(self.p['r'])
        self.af = float(self.p['alpha_f'])
        self.ab = float(self.p['alpha_b'])
        
        # Extract DC motor electromechanical constants
        self.R = float(self.p['R'])
        self.Lm = float(self.p['Lm'])
        self.Kt = float(self.p['Kt'])
        self.Kb = float(self.p['Kb'])
        
        # Actuator and electrical operating limits
        self.V_max = float(self.p.get('V_max', 12.0))
        self.I_max = float(self.p.get('I_max', 20.0))
        
        # State and input dimensions
        self.n_x = 10
        self.n_u = 4
        
        # 1. Asymmetric 4-omniwheel kinematic projection matrix H
        self.H = np.array([
            [-np.sin(self.af),  np.cos(self.af), self.Rb],
            [-np.sin(self.af), -np.cos(self.af), self.Rb],
            [ np.sin(self.ab), -np.cos(self.ab), self.Rb],
            [ np.sin(self.ab),  np.cos(self.ab), self.Rb]
        ])
        
        # 2. Chassis inverse mass/inertia matrix
        self.M_inv = np.diag([1.0 / self.Mr, 1.0 / self.Mr, 1.0 / self.Jzz])
        
        # 3. Pre-compute constant coupling submatrices (computed ONCE at init)
        self.A_vi = (self.Kt / self.r) * (self.M_inv @ self.H.T)       # (3, 4) Current -> Accel
        self.A_iv = -(self.Kb / (self.Lm * self.r)) * self.H           # (4, 3) Back-EMF -> di/dt
        self.A_ii = -(self.R / self.Lm) * np.eye(4)                     # (4, 4) Ohmic decay
        
        # 4. Input matrix Bc (constant 10x4)
        self.B_c = np.zeros((self.n_x, self.n_u))
        self.B_c[6:10, :] = (1.0 / self.Lm) * np.eye(self.n_u)
        
    def get_rotation_matrix(self, theta: float) -> np.ndarray:
        """Evaluates planar 3x3 coordinate rotation matrix R(theta). GLOBAL FRAME: the plant's rotation."""
        return OmniRobotSimulationODESolve._rotation(theta)
        
    def get_continuous_matrices(self, theta: float) -> tuple:
        """
        Evaluates the continuous state-space matrices Ac(theta) and Bc.
        
        Parameters:
            theta (float): Current chassis heading angle in radians.
            
        Returns:
            A_c (10, 10) ndarray, B_c (10, 4) ndarray
        """
        R_theta = self.get_rotation_matrix(theta)
        
        A_c = np.zeros((self.n_x, self.n_x))
        # GLOBAL FRAME: the velocity state is global. The wheels push along body directions, so the current-to-force
        # coupling is rotated into the global frame, and the back-EMF uses the local velocity R(theta)^T v_global.
        # No turning term is needed: Newton's law holds in the global frame as it is.
        A_c[:3, 3:6] = np.eye(3)                   # Kinematics: d(pos)/dt = v_global
        A_c[3:6, 6:10] = R_theta @ self.A_vi       # Kinetics:   d(v_global)/dt = R(theta) A_vi i
        A_c[6:10, 3:6] = self.A_iv @ R_theta.T     # Electrical: di/dt = A_iv R(theta)^T v_global - (R/Lm)*i + (1/Lm)*V
        A_c[6:10, 6:10] = self.A_ii
        
        return A_c, self.B_c
        
    def get_discrete_matrices(self, theta: float, dt: float = 0.02) -> tuple:
        """
        Computes exact Zero-Order Hold (ZOH) discrete state-space matrices (A_d, B_d)
        for heading angle theta via Van Loan's block matrix exponential.
        
        Parameters:
            theta (float): Operating heading angle in radians.
            dt (float): Discrete control period (default: 0.02s = 50 Hz).
            
        Returns:
            A_d (10, 10) ndarray, B_d (10, 4) ndarray
        """
        A_c, B_c = self.get_continuous_matrices(theta)
        
        M_block = np.zeros((self.n_x + self.n_u, self.n_x + self.n_u))
        M_block[:self.n_x, :self.n_x] = A_c * dt
        M_block[:self.n_x, self.n_x:] = B_c * dt
        
        exp_M = la.expm(M_block)
        A_d = exp_M[:self.n_x, :self.n_x]
        B_d = exp_M[:self.n_x, self.n_x:]
        
        return A_d, B_d



In [35]:
import numpy as np
import scipy.linalg as la
import cvxpy as cp

class PoseVelocityEstimator:
    """
    Reconstructs the GLOBAL chassis velocity by differentiating successive global pose measurements.
    Accounts for [-pi, pi] angular phase wrapping. GLOBAL FRAME: kept in the global frame (the MPC's velocity state).
    """
    def __init__(self, dt: float = 0.02):
        self.dt = float(dt)
        self.pos_past = None
        
    def reset(self, initial_pos: np.ndarray = None):
        self.pos_past = initial_pos.copy() if initial_pos is not None else None
        
    def estimate(self, current_pos_global: np.ndarray) -> np.ndarray:
        """
        Parameters:
            current_pos_global (3,): [x, y, theta] in meters, radians
        Returns:
            v_global (3,): [vx_g, vy_g, omega_yaw] in m/s, rad/s (GLOBAL FRAME)
        """
        if self.pos_past is None:
            self.pos_past = current_pos_global.copy()
            return np.zeros(3)
            
        # 1. Finite difference in world frame
        dp = current_pos_global[:2] - self.pos_past[:2]
        v_global_xy = dp / self.dt
        
        # 2. Angular difference with [-pi, pi] phase unwrapping
        d_theta = ((current_pos_global[2] - self.pos_past[2] + np.pi) % (2.0 * np.pi)) - np.pi
        omega_yaw = d_theta / self.dt
        
        # 3. GLOBAL FRAME: no rotation into the local frame; the MPC's velocity state is global

        # Update historical pose memory
        self.pos_past = current_pos_global.copy()

        v_global = np.array([v_global_xy[0], v_global_xy[1], omega_yaw])
        return v_global


class EncoderKinematicEstimator:
    """
    Reconstructs local chassis velocity from 4 wheel encoder speeds via Moore-Penrose pseudoinverse.
    Assumes nominal rolling contact kinematics without gross tire spinout.
    """
    def __init__(self, simulation_params: dict):
        self.r = float(simulation_params['r'])
        self.Rb = float(simulation_params['Rb'])
        self.af = float(simulation_params['alpha_f'])
        self.ab = float(simulation_params['alpha_b'])
        
        # Asymmetric 4-omniwheel kinematic projection matrix H (4x3)
        self.H = np.array([
            [-np.sin(self.af),  np.cos(self.af), self.Rb],
            [-np.sin(self.af), -np.cos(self.af), self.Rb],
            [ np.sin(self.ab), -np.cos(self.ab), self.Rb],
            [ np.sin(self.ab),  np.cos(self.ab), self.Rb]
        ])
        
        # Pre-compute Moore-Penrose pseudoinverse (H^T @ H)^-1 @ H^T (3x4)
        self.H_pinv = np.linalg.pinv(self.H)
        
    def reset(self):
        pass
        
    def estimate(self, current_omega: np.ndarray) -> np.ndarray:
        """
        Parameters:
            current_omega (4,): [w1, w2, w3, w4] wheel angular speeds in rad/s
        Returns:
            v_local_odom (3,): [vx_r, vy_r, omega_yaw] in m/s, rad/s
        """
        return self.r * (self.H_pinv @ current_omega)



class MPCControllerSimple:
    """
    Condensed 10-State Convex Model Predictive Controller (MPC) for 4-Omniwheel Robot.
    
    Operates at a 50 Hz discrete execution rate (dt = 0.02s) to track time-indexed 
    trajectory setpoints over a look-ahead prediction horizon of K steps.
    
    Delegates system discretization and kinematic linearization to the internal 
    OmniRobotNominalModel class, incorporating:
      - Position/heading error penalties
      - Velocity tracking error stage costs
      - Actuator terminal voltage magnitude and slew-rate bounds (±12V, ±5V/step)
      - Motor driver over-current hardpoint bounds (±20A)
      - Successive linearization along the predicted heading trajectory theta_pred[j] (RTI)
      - Disciplined Parameterized Programming (DPP) using cp.sum_squares for ultra-fast warm-started QP execution
    """
    def __init__(self, simulation_params: dict,
                 horizon_K: int = 8,
                 dt: float = 0.02,
                 q_pose: tuple = (100.0, 100.0, 60.0),
                 q_vel: tuple = (15.0, 15.0, 8.0),
                 r_voltage: float = 0.015,
                 r_slew: float = 0.008,
                 voltage_limit: float = 12.0,
                 current_limit: float = 20.0,
                 delta_v_max: float = 5.0):
        """
        Subfunction 1: Initializes controller parameters, weighting matrices, and physical bounds.
        Delegates physics to OmniRobotNominalModel and pre-compiles the solver graph via initializeOptimizationProblem().
        """
        self.p = simulation_params.copy()
        self.K = int(horizon_K)
        self.dt = float(dt)
        self.voltage_limit = float(voltage_limit)
        self.current_limit = float(current_limit)
        self.delta_v_max = float(delta_v_max)
        
        # Instantiate the nominal prediction model
        self.nominal_model = OmniRobotNominalModel(self.p)
        self.n_x = self.nominal_model.n_x  # 10 states [p(3), v(3), i(4)]
        self.n_u = self.nominal_model.n_u  # 4 inputs  [V1, V2, V3, V4]
        self.I_max = self.nominal_model.I_max

        # Instantiate Estimator Classes
        self.pose_estimator = PoseVelocityEstimator(dt=self.dt)
        self.encoder_estimator = EncoderKinematicEstimator(self.p)
        
        # Quadratic weighting matrices (matrix square roots for strict DPP compliance with cp.sum_squares)
        self.Q_pos_sqrt = np.diag(np.sqrt(q_pose))
        self.Q_vel_sqrt = np.diag(np.sqrt(q_vel))
        self.R_sqrt = np.sqrt(r_voltage) * np.eye(self.n_u)
        self.R_slew_sqrt = np.sqrt(r_slew) * np.eye(self.n_u)
        
        # Terminal weights (soft tracking penalty ensures 100% feasibility)
        self.P_pos_sqrt = np.diag(np.sqrt(10.0 * np.array(q_pose)))
        self.P_vel_sqrt = np.diag(np.sqrt(5.0 * np.array(q_vel)))
        
        # Pre-compile the symbolic optimization problem ONCE at startup
        self.initializeOptimizationProblem()
        
        # Initialize memory and solver telemetry
        self.reset()


    def initializeOptimizationProblem(self):
        """
        Subfunction 2: Compiles the symbolic CVXPY optimization problem ONCE using cp.Parameter.
        Enables in-place parameter updates and warm-started OSQP execution at 50 Hz.
        Enforces box constraints on both actuator voltages and motor armature currents.
        """
        K = self.K
        n_x = self.n_x
        n_u = self.n_u
        
        # Optimization Decision Variables (allocated ONCE)
        self.X_var = cp.Variable((n_x, K + 1))
        self.U_var = cp.Variable((n_u, K))
        
        # Optimization Parameters (updated in-place at each 50 Hz cycle)
        self.x0_param = cp.Parameter((n_x,))
        self.A_params = [cp.Parameter((n_x, n_x)) for _ in range(K)]
        self.B_params = [cp.Parameter((n_x, n_u)) for _ in range(K)]
        self.pos_ref_param = cp.Parameter((K + 1, 3))
        self.vel_ref_param = cp.Parameter((K + 1, 3))
        self.u_prev_param = cp.Parameter((n_u,))
        
        cost = 0
        constraints = [self.X_var[:, 0] == self.x0_param]
        
        for j in range(K):
            # Tracking error stage penalties via cp.sum_squares (strict DPP compliance)
            e_pos_j = self.X_var[:3, j] - self.pos_ref_param[j]
            e_vel_j = self.X_var[3:6, j] - self.vel_ref_param[j]
            cost += cp.sum_squares(self.Q_pos_sqrt @ e_pos_j)
            cost += cp.sum_squares(self.Q_vel_sqrt @ e_vel_j)
            cost += cp.sum_squares(self.R_sqrt @ self.U_var[:, j])
            
            # Slew rate penalty and traction protection bounds
            if j == 0:
                du = self.U_var[:, j] - self.u_prev_param
            else:
                du = self.U_var[:, j] - self.U_var[:, j - 1]
            cost += cp.sum_squares(self.R_slew_sqrt @ du)
            constraints += [
                du >= -self.delta_v_max,
                du <=  self.delta_v_max
            ]
                
            # Step-varying discrete linear dynamics constraint (from OmniRobotNominalModel)
            constraints += [
                self.X_var[:, j + 1] == self.A_params[j] @ self.X_var[:, j] + self.B_params[j] @ self.U_var[:, j]
            ]
            
            # Actuator terminal voltage saturation box limits: ±12V
            constraints += [
                self.U_var[:, j] >= -self.voltage_limit,
                self.U_var[:, j] <=  self.voltage_limit
            ]
            
            # Motor driver over-current protection limits: ±20A
            constraints += [
                self.X_var[6:10, j] >= -self.current_limit,
                self.X_var[6:10, j] <=  self.current_limit
            ]
            
        # Soft terminal tracking costs via cp.sum_squares
        e_pos_K = self.X_var[:3, K] - self.pos_ref_param[K]
        e_vel_K = self.X_var[3:6, K] - self.vel_ref_param[K]
        cost += cp.sum_squares(self.P_pos_sqrt @ e_pos_K)
        cost += cp.sum_squares(self.P_vel_sqrt @ e_vel_K)
        
        # Terminal current bound
        constraints += [
            self.X_var[6:10, K] >= -self.current_limit,
            self.X_var[6:10, K] <=  self.current_limit
        ]
        
        # Pre-compile the problem
        self.problem = cp.Problem(cp.Minimize(cost), constraints)

    # Alias for method naming flexibility
    initialize_optimization_problem = initializeOptimizationProblem

    def reset(self):
        """
        Subfunction 3: Resets operational memory and solver status indicators.
        """
        self.u_prev = np.zeros(self.n_u)
        self.last_status = "unsolved"
        self.fallback_used = False
        self.last_X_opt = None
        self.theta_pred = np.zeros(self.K)

    def get_discrete_model(self, theta: float):
        """
        Subfunction 4: Computes discrete state-space matrices (A_d, B_d) at heading theta
        by delegating directly to the OmniRobotNominalModel.
        """
        return self.nominal_model.get_discrete_matrices(theta, self.dt)

    def compute_optimal_u0(self, measured_State: OmniRobotObservableState,
                           path_ref_horizon: np.ndarray, vel_ref_horizon: np.ndarray,
                           return_info: bool = False):
        """
        Subfunction 5: Fast 50 Hz execution interface. Updates pre-allocated parameters 
        in-place and solves the pre-compiled QP in real time.
        
        Parameters:
            measured_State (OmniRobotObservableState): Current robot state with position and velocity.
                measured_pose (3,) ndarray: [x, y, theta] actual world coordinates.
                measured_v_global (3,) ndarray: [vx_g, vy_g, omega_yaw] actual global velocities (GLOBAL FRAME).
                measured_currents (4,) ndarray, optional: Motor armature currents [i1..4] (defaults to steady-state estimation if None).
                            
            path_ref_horizon (K+1, 3) ndarray: Desired poses [x_d, y_d, theta_d] over horizon K.
            vel_ref_horizon (K+1, 3) ndarray: Desired GLOBAL velocities [vx_d, vy_d, omega_d] over horizon K (GLOBAL FRAME).
            return_info (bool): Whether to return a diagnostic telemetry dictionary.
            
        Returns:
            u_0 (4,) ndarray: Commanded motor terminal voltages [V1, V2, V3, V4].
        """
        # convert measured state to numpy arrays for consistency
        measured_pose = np.asarray(measured_State.pos_global, dtype=float)
        measured_currents = np.asarray(measured_State.I, dtype=float) if measured_State.I is not None else None

        # Estimate velocities using the two estimator models
        v_est_pose = self.pose_estimator.estimate(measured_State.pos_global)
        v_est_encoder = self.encoder_estimator.estimate(measured_State.omega)

        # Use pose-differentiated velocity as the active chassis velocity (GLOBAL FRAME: global)
        measured_v_global = v_est_pose.copy()
                    
        # Compute contact patch slip diagnostic: v_slip = H @ v_local - r * omega
        # GLOBAL FRAME: the wheels are in the body frame, so the global velocity is rotated: v_local = R(theta)^T v_global
        v_contact = self.encoder_estimator.H @ (OmniRobotSimulationODESolve._rotation(measured_pose[2]).T @ v_est_pose)
        v_slip = v_contact - self.encoder_estimator.r * measured_State.omega

        # there are two ways to measure the local velocity, either from the global position or from the wheel speeds
        # the diffrene between the wheel speed and position velocity calculation will be the friction disturbances
        # initially we will not calculate the friction disturbances with an kalman observer 
                                      
        # Unwrap heading angle relative to reference horizon to prevent 2*pi phase jumps
        ref_theta0 = path_ref_horizon[0, 2]
        unwrapped_theta = (measured_pose[2] - ref_theta0 + np.pi) % (2.0 * np.pi) - np.pi + ref_theta0

        # create the state vector
        x0_10 = np.concatenate([[measured_pose[0], measured_pose[1], unwrapped_theta],
                                 measured_v_global, measured_currents ])
        
        # 1. Update predicted heading trajectory theta_pred
        theta_pred = np.zeros(self.K)
        theta_pred[0] = unwrapped_theta
        if self.last_X_opt is not None:
            bias = unwrapped_theta - self.last_X_opt[2, 1]
            for j in range(1, self.K):
                theta_pred[j] = self.last_X_opt[2, j + 1] + bias
        else:
            for j in range(1, self.K):
                theta_pred[j] = unwrapped_theta + j * self.dt * measured_v_global[2]
        self.theta_pred = theta_pred
        
        # 2. Evaluate and set step-varying (A_d, B_d) via OmniRobotNominalModel
        for j in range(self.K):
            Ad_j, Bd_j = self.nominal_model.get_discrete_matrices(self.theta_pred[j], self.dt)
            self.A_params[j].value = Ad_j
            self.B_params[j].value = Bd_j
            
        # 3. Update CVXPY parameters in-place (ZERO dynamic memory allocations!)
        self.x0_param.value = x0_10
        self.pos_ref_param.value = path_ref_horizon
        self.vel_ref_param.value = vel_ref_horizon
        self.u_prev_param.value = self.u_prev
        
        # 4. Solve pre-compiled QP with warm-start
        self.problem.solve(solver=cp.OSQP, eps_abs=1e-3, eps_rel=1e-3, max_iter=4000, warm_start=True, verbose=False)
        
        self.last_status = self.problem.status
        self.fallback_used = (self.problem.status not in ["optimal", "optimal_inaccurate"] or self.U_var.value is None)
        
        if self.fallback_used:
            u_0 = self.u_prev.copy()
        else:
            u_0 = self.U_var.value[:, 0].copy()
            self.last_X_opt = self.X_var.value.copy()
            
        self.u_prev = u_0.copy()
        if return_info:
            return u_0, {
                        'optimal': not self.fallback_used,
                        'status': self.last_status,
                        'fallback': self.fallback_used,
                        'theta_pred': self.theta_pred.copy(),
                        'v_est_pose': v_est_pose.copy(),
                        'v_est_encoder': v_est_encoder.copy(),
                        'v_slip': v_slip.copy()
                    }
        return u_0



## `OmniRobotLinearizedModel`: the linearized MPC model, built step by step (added in this copy)

An MPC model from the plant's own equations, built one step at a time. Dana's classes above are not changed by this (they only have the global-velocity change); later steps reuse her `MPCControllerSimple` through a subclass. The checks are in `ControllerAnalysis_Estimator.ipynb`, where the plant is loaded.

| Step | What | Status |
|---|---|---|
| 1 | The model's equations, $\dot{\mathbf{x}} = f(\mathbf{x}, \mathbf{u})$, checked against the plant | done |
| 2 | Linearizing: $\mathbf{A} = \partial f / \partial \mathbf{x}$, $\mathbf{B} = \partial f / \partial \mathbf{u}$ | done |
| 3 | The 20 ms model $\mathbf{x}_{j+1} = \mathbf{A}_d \mathbf{x}_j + \mathbf{B}_d \mathbf{u}_j + \mathbf{c}_d$, relinearized 4 times per step (every 5 ms) | done |
| 4 | The MPC: a subclass of Dana's `MPCControllerSimple` | done |
| 5 | The estimator | |
| 6 | Dana's runner: against PID and the simple MPC | |

### Step 1: the model's equations

#### Why a new model

Dana's `OmniRobotNominalModel` assumes every wheel puts all its motor torque into the floor, $F_i = K_t i_i / r$. It has no friction and no wheel speeds, so it cannot tell when a wheel will slide. This model keeps the plant's friction: the wheel speeds $\omega$ and the bristle deflections $z$ are states, so the MPC can predict sliding.

#### The state

| Index | State | Units | Dana's model |
|---|---|---|---|
| 0:3 | $X, Y, \theta$: pose, global frame | m, m, rad | same |
| 3:6 | $\dot X, \dot Y, \dot\theta$: velocity, **global frame** | m/s, m/s, rad/s | same (her model is global in this copy too) |
| 6:10 | $i_1 \dots i_4$: motor currents | A | same |
| 10:14 | $\omega_1 \dots \omega_4$: wheel speeds | rad/s | **new** |
| 14:18 | $z_1 \dots z_4$: bristle deflections (fraction of $\delta$) | - | **new** |

The input is the four motor voltages $\mathbf{u} = [V_1 \dots V_4]$, as in Dana's model. The positions in $\mathbf{x}$ are the same as hers (pose 0:3, velocity 3:6, currents 6:10), so her MPC's indexing still works in Step 4.

**One velocity state: the global one,** as in the plant (Step 5.9 advances the global velocity, where Newton's law holds as it is). The velocity along the robot's axes, $\mathbf{v}_{local} = \mathbf{R}(\theta)^\top \dot{\mathbf{X}}_g$, is worked out where it is needed (the wheel slip and the roller slip); it is not a state.

**Why velocity is a state at all.** The chassis has mass: force sets the acceleration, so the velocity carries the momentum, and nothing else in the state gives it. Without slip it could be worked out from the wheel speeds, which is why Dana's model needs no wheel speeds. With slip the chassis and the wheels move independently, and the slip $v_s = \mathbf{H}_{drive}\mathbf{v}_{local} - r\omega$ that drives the friction needs both.

#### The equations (each from the plant step in brackets)

* **Velocity along the robot's axes:** $\mathbf{v}_{local} = \mathbf{R}(\theta)^\top \dot{\mathbf{X}}_g$
* **Floor force** [Step 5.6]: $F_i = -\mu_s F_{n,i} z_i$
* **Roller force** [Step 5.8]: $F_{a,i} = -F_{n,i}\,\mu_{a,i}\,\frac{2}{\pi}\arctan(k_a v_{a,i})$, $v_{a,i} = (\mathbf{H}_{axial}\mathbf{v}_{local})_i$ (zero when `roller_friction` is off)
* **Chassis, Newton's law in the global frame** [Step 5.9]: $\ddot{\mathbf{X}}_g = \mathbf{M}^{-1}\mathbf{R}(\theta)\left(\mathbf{H}_{drive}^\top \mathbf{F} + \mathbf{H}_{axial}^\top \mathbf{F}_a\right)$
* **Pose**: $\frac{d}{dt}[X, Y, \theta] = \dot{\mathbf{X}}_g$
* **Motor** [Step 5.1]: $L_m \dot i_i = V_i - R\,i_i - K_b\,\omega_i$
* **Wheel** [Steps 5.3, 5.7]: $J_w \dot\omega_i = K_t i_i - T_{roll,i} - r F_i$, with $T_{roll,i} = C_{rr} F_{n,i}\, r \tanh(\omega_i / \varepsilon)$
* **Bristle** [Step 5.5]: $\delta\,\dot z_i = v_{s,i} - d_i z_i$, with the drive slip $v_{s,i} = (\mathbf{H}_{drive}\mathbf{v}_{local})_i - r\,\omega_i$

($\mathbf{M}^{-1}\mathbf{R} = \mathbf{R}\mathbf{M}^{-1}$ here, because $\mathbf{M} = \mathrm{diag}(M_r, M_r, J_{zz})$ has the same mass in x and y.)

#### The loads, and the one value held

* **The wheel loads $F_{n,i}$** [Step 5.2] are worked out from the state, inside $f$ (`loads(x)`): the static split plus the pitch and roll load transfer of this state's own acceleration. The acceleration depends on the friction forces and the forces are proportional to the loads, so the loads come from a small 4x4 linear solve, $(\mathbf{I} - \mathbf{L}\mathbf{M}^{-1}\mathbf{K})\,\mathbf{F}_n = \mathbf{F}_{n,static}$. The plant uses the previous 1 ms step's acceleration instead; at 1 ms the two are nearly the same. Because the loads are part of $f$, the linearization includes them.
* **The bristle relax rate $d_i = \frac{\mu_s}{\mu_{d,i}}\sqrt{v_{s,i}^2 + v_c^2}$** [Steps 5.4, 5.5] is held: worked out at each linearization point (`held_values(x)`), not differentiated (Step 2 says why).

**No noise in the model.** The plant's load and friction noise ($w_F$, $w_\mu$) are left out: the model uses the nominal $\mu_s$, $\mu_k$.

### Step 2: linearizing, $\mathbf{A} = \partial f / \partial \mathbf{x}$ and $\mathbf{B} = \partial f / \partial \mathbf{u}$

Near a point $(\bar{\mathbf{x}}, \bar{\mathbf{u}})$ the model is approximated by a straight line:
$$f(\mathbf{x}, \mathbf{u}) \approx f(\bar{\mathbf{x}}, \bar{\mathbf{u}}) + \mathbf{A}\,(\mathbf{x} - \bar{\mathbf{x}}) + \mathbf{B}\,(\mathbf{u} - \bar{\mathbf{u}})$$
Column $i$ of $\mathbf{A}$ says how every rate changes per unit change of state $i$. `jacobians()` works it out by nudging state $i$ up and down by a small $h$: $\mathbf{A}_{:,i} = \big(f(\bar{\mathbf{x}} + h\,\mathbf{e}_i) - f(\bar{\mathbf{x}} - h\,\mathbf{e}_i)\big) / 2h$, and the same for each voltage in $\mathbf{B}$. Everything in $f$ is differentiated (the loads, the rotation $\mathbf{R}(\theta)$, the rollers, the rolling drag) except the relax rate $d$.

Many entries can be written down by hand from the equations (with the loads held, so the formulas stay short), which is how the check tests it:

| Entry | From | Value |
|---|---|---|
| $\partial \dot{\mathbf{X}} / \partial \dot{\mathbf{X}}_g$ | pose | $\mathbf{I}$ |
| $\partial \dot i / \partial V$ | motor | $1/L_m$ |
| $\partial \dot i / \partial i$ | motor | $-R/L_m$ |
| $\partial \dot i / \partial \omega$ | motor | $-K_b/L_m$ |
| $\partial \dot\omega / \partial i$ | wheel | $K_t/J_w$ |
| $\partial \dot\omega / \partial z$ | wheel, with $F = -\mu_s F_n z$ | $+r\,\mu_s F_n / J_w$ |
| $\partial \dot z / \partial \omega$ | bristle | $-r/\delta$ |
| $\partial \dot z / \partial z$ | bristle | $-d/\delta$ |

**Why the relax rate $d$ is held, not differentiated.** It is worked out again at every linearization point (every 5 ms in Step 3), so it still follows the slip; it is only held *within* a linearization. $d = \frac{\mu_s}{\mu_d}\sqrt{v_s^2 + v_c^2}$ has a sharp corner at zero slip: $v_c$ is only 0.1 mm/s, so its slope flips from about $-\mu_s/\mu_d$ to $+\mu_s/\mu_d$ across ±0.1 mm/s. A slope taken there means nothing a few milliseconds later; differentiating it (in the 11-09 tests) gave growing modes up to +390 /s.

What holding it costs: within a sub-step the linear model does not know that more slip relaxes the bristles more (which is what caps the force at $\mu_d F_n$ while sliding).

**The loads are differentiated.** An earlier version held them too, taking each sub-step's loads from the previous sub-step's acceleration. That delayed loop (acceleration, loads, acceleration) made one 0.4 s prediction blow up (a state grew about 2000 times in one 20 ms step). Working the loads out from the state inside $f$ removed it.

### Step 3: the 20 ms model, $\mathbf{x}_{j+1} = \mathbf{A}_d\mathbf{x}_j + \mathbf{B}_d\mathbf{u}_j + \mathbf{c}_d$

The MPC needs one linear map per 20 ms controller step. The bristles are too stiff to step simply, so each 20 ms step is split into **4 backward-Euler sub-steps of 5 ms**, the same kind of step the plant takes:

$$\mathbf{x}_{q+1} = \mathbf{x}_q + h\,f(\mathbf{x}_{q+1}), \qquad f(\mathbf{x}) \approx \bar f + \mathbf{A}(\mathbf{x} - \bar{\mathbf{x}}) + \mathbf{B}(\mathbf{u} - \bar{\mathbf{u}})$$
$$\Rightarrow\ \mathbf{x}_{q+1} = \mathbf{a}\,\mathbf{x}_q + \mathbf{b}\,\mathbf{u} + \mathbf{c}, \qquad \mathbf{a} = (\mathbf{I} - h\mathbf{A})^{-1},\ \ \mathbf{b} = h\,\mathbf{a}\mathbf{B},\ \ \mathbf{c} = h\,\mathbf{a}\left(\bar f - \mathbf{A}\bar{\mathbf{x}} - \mathbf{B}\bar{\mathbf{u}}\right)$$

Each sub-step is **linearized again at the state the previous one reached** (`substep_map`), so the friction state, the loads and the rotation follow the robot within the 20 ms. The four maps are chained into one (`step_model`): $\mathbf{A}_d = \mathbf{a}_4\mathbf{a}_3\mathbf{a}_2\mathbf{a}_1$, with $\mathbf{B}_d$, $\mathbf{c}_d$ collected the same way.

**The loads** are part of $f$, so each sub-step's linearization includes the load transfer.

### Step 4: the MPC, `MPCControllerLinearized`

A subclass of Dana's `MPCControllerSimple`. Her cost, weights, limits and solver are unchanged. What is different:
1. **The model:** the 18-state linearized model instead of her 10-state model. The first 10 states are in her order (pose, global velocity, currents), so her cost and current limits apply to the same rows.
2. **The $\mathbf{c}_j$ term** in each step's model: $\mathbf{X}_{j+1} = \mathbf{A}_j\mathbf{X}_j + \mathbf{B}_j\mathbf{U}_j + \mathbf{c}_j$ (the linearization is around a point, not the origin).
3. **Where it linearizes:** along its last plan. Each 20 ms the last plan's voltages and sub-step maps are moved one step on and carry the current state along them; that gives the 8 x 4 = 32 sub-step points, which are all linearized in **one batch** (`linearize_along`) and chained into one $(\mathbf{A}_j, \mathbf{B}_j, \mathbf{c}_j)$ per horizon step. This is how JP3 did it on 11-09: one numpy call for every nudged point of every sub-step (32 x 45 points) instead of 1440 separate calls, which is what keeps it inside the 20 ms budget.
4. **The starting state:** pose, wheel speeds and currents measured; the global velocity from her `PoseVelocityEstimator`; the bristle state $z$ from each wheel's torque balance with the measured current and wheel speed, $F = (K_t I - T_{roll} - J_w\dot\omega)/r$, $z = -F/(\mu_s F_n)$. With `use_true_state=True` it is given the plant's true state instead (an ideal observer), to separate the model from the estimation.


In [ ]:
import numpy as np


class OmniRobotLinearizedModel:
    """
    The plant's equations as one continuous model, dx/dt = f(x, u), for the MPC to linearize (Step 2 on).

    State x (18). Same positions as Dana's OmniRobotNominalModel for the first 10:
        x[0:3]   = [X, Y, theta]              pose, global frame (m, m, rad)
        x[3:6]   = [X_dot, Y_dot, theta_dot]  velocity, GLOBAL frame (m/s, m/s, rad/s)   same as Dana's (global)
        x[6:10]  = [i_1 .. i_4]               motor currents (A)
        x[10:14] = [omega_1 .. omega_4]       wheel speeds (rad/s)                       NEW: Dana's model has none
        x[14:18] = [z_1 .. z_4]               bristle deflections (fraction of delta)    NEW: Dana's model has none
    Input u = [V_1 .. V_4] (V).

    Every method works on MANY points at once (the "_batch" methods: one row per point), as JP3 did on 11-09:
    evaluating f for all the nudged points of all the sub-steps in one numpy call is what keeps the linearization
    fast. The methods without "_batch" do one point (they call the batched one with one row); the checks use them.

    Subfunctions:
      1. __init__: parameters, the fixed geometry and loads from the plant class, the load transfer matrix
      2. rotation / rotation_batch: R(theta), the plant's rotation (for one angle, or many at once)
      3. local_velocity(_batch): the velocity along the robot's axes, R(theta)^T X_dot_g
      4. roller_factor_batch: roller friction per unit load, from the axial slip (plant Step 5.8)
      5. loads(_batch): the wheel loads with the load transfer of the state's own acceleration (plant Step 5.2)
      6. held_values(_batch): the bristle relax rate d (the one value held)
      7. f(_batch): the state derivatives
      8. jacobians(_batch): A = df/dx and B = df/du (Step 2)
      9. substep_map(s_batch): one backward-Euler sub-step, linearized at a point (Step 3)
      10. step_model: one controller step, relinearized at each sub-step, one point after another (Step 3, checks)
      11. linearize_along: every sub-step point of a plan in one batch, chained into one model per step (Step 3, MPC)
      12. from_plant: the model's state from a plant state
    """
    # Where each part of the state sits in x
    POSE = slice(0, 3)
    VELOCITY = slice(3, 6)
    CURRENTS = slice(6, 10)
    WHEELS = slice(10, 14)
    BRISTLES = slice(14, 18)
    n_x = 18
    n_u = 4

    def __init__(self, simulation_params: dict, plant_class=None):
        self.p = simulation_params
        if plant_class is None:
            plant_class = OmniRobotSimulationODESolve

        # Fixed values only (geometry, static loads, load transfer gains), from the plant's own __init__,
        # so the model and the plant cannot disagree on them. No state is read from it.
        geometry = plant_class(simulation_params)
        self.H_drive = geometry.H_drive              # 4x3: drive speeds = H_drive v_local
        self.H_axial = geometry.H_axial              # 4x3: axial (roller) speeds = H_axial v_local
        self.M_inv = geometry.M_inv                  # 3x3: diag(1/Mr, 1/Mr, 1/Jzz)
        self.F_n_static = np.array([geometry.F_n_front_static, geometry.F_n_front_static,
                                    geometry.F_n_rear_static, geometry.F_n_rear_static])
        self.pitch_gain = geometry.pitch_gain
        self.roll_gain_front = geometry.roll_gain_front
        self.roll_gain_rear = geometry.roll_gain_rear

        # Load moved per unit acceleration along the robot's axes: rows wheels 1 FR, 2 FL, 3 RL, 4 RR;
        # columns [right, forward, yaw] (plant Step 5.2: accelerating right or forward unloads wheel 1)
        self.LT = np.array([
            [-self.roll_gain_front, -self.pitch_gain, 0.0],     # 1 FR
            [self.roll_gain_front, -self.pitch_gain, 0.0],      # 2 FL
            [self.roll_gain_rear, self.pitch_gain, 0.0],        # 3 RL
            [-self.roll_gain_rear, self.pitch_gain, 0.0],       # 4 RR
        ])

    # =======================================================================
    # Frames
    # =======================================================================

    @staticmethod
    def rotation(theta: float) -> np.ndarray:
        """R(theta): local frame -> global frame. Its transpose goes back. GLOBAL FRAME: the plant's rotation."""
        R_theta = OmniRobotSimulationODESolve._rotation(theta)
        return R_theta

    @staticmethod
    def rotation_batch(theta: np.ndarray) -> np.ndarray:
        """R(theta) for many angles at once, (N, 3, 3): the same matrix as the plant's _rotation, one per row."""
        c = np.cos(theta)
        s = np.sin(theta)
        R_theta = np.zeros((len(theta), 3, 3))
        R_theta[:, 0, 0] = c
        R_theta[:, 0, 1] = -s
        R_theta[:, 1, 0] = s
        R_theta[:, 1, 1] = c
        R_theta[:, 2, 2] = 1.0
        return R_theta

    def local_velocity_batch(self, X: np.ndarray) -> np.ndarray:
        """Velocity along the robot's axes for each row: v_local = R(theta)^T X_dot_g, (N, 3)."""
        R_theta = self.rotation_batch(X[:, 2])
        v_local = np.einsum('nji,nj->ni', R_theta, X[:, self.VELOCITY])     # R^T v, row by row
        return v_local

    def local_velocity(self, x: np.ndarray) -> np.ndarray:
        """Velocity along the robot's axes at one point."""
        v_local = self.local_velocity_batch(x[None])[0]
        return v_local

    # =======================================================================
    # Forces and loads
    # =======================================================================

    def roller_factor_batch(self, v_local: np.ndarray) -> np.ndarray:
        """
        Roller (axial) friction per unit load on each wheel, from its axial slip v_slip_a = H_axial v_local (plant
        Step 5.8): F_a = roller_factor * F_n, (N, 4). Zero when roller_friction is off.
        """
        p = self.p
        if not p['roller_friction']:
            return np.zeros((len(v_local), 4))
        v_slip_a = v_local @ self.H_axial.T
        mu_a = p['mu_a_k'] + (p['mu_a_s'] - p['mu_a_k']) * np.exp(-p['alpha_a'] * np.abs(v_slip_a))
        factor = -mu_a * (2.0 / np.pi) * np.arctan(p['roller_smoothing'] * v_slip_a)
        return factor

    def loads_batch(self, X: np.ndarray) -> np.ndarray:
        """
        The wheel loads F_n for each row, with the pitch and roll load transfer of that state's own acceleration
        (plant Step 5.2 uses the previous 1 ms step's acceleration; here it is the same instant, solved together).

        The acceleration depends on the friction forces, and the forces are proportional to the loads:
            F_n = F_static + LT a_local,   a_local = M^-1 (H_drive^T F + H_axial^T F_a)   (along the robot's axes)
            F = -mu_s F_n z,   F_a = roller_factor F_n
        so  (I - LT M^-1 K) F_n = F_static,   K = H_drive^T diag(-mu_s z) + H_axial^T diag(roller_factor)
        One 4x4 solve per row, all rows in one call. Because the loads are worked out inside f, the Jacobians
        differentiate them.
        """
        p = self.p
        z = X[:, self.BRISTLES]
        v_local = self.local_velocity_batch(X)

        drive_per_load = -p['mu_s'] * z                                    # F = drive_per_load * F_n
        axial_per_load = self.roller_factor_batch(v_local)                 # F_a = axial_per_load * F_n
        K = self.H_drive.T[None] * drive_per_load[:, None, :] + self.H_axial.T[None] * axial_per_load[:, None, :]   # (N, 3, 4)

        system = np.eye(4)[None] - self.LT[None] @ (self.M_inv[None] @ K)  # (N, 4, 4)
        right_side = np.broadcast_to(self.F_n_static, (len(X), 4))[..., None]
        F_n = np.linalg.solve(system, right_side)[..., 0]
        F_n = np.maximum(0.0, F_n)                                         # a wheel cannot pull on the floor
        return F_n

    def loads(self, x: np.ndarray) -> np.ndarray:
        """The wheel loads at one point."""
        F_n = self.loads_batch(x[None])[0]
        return F_n

    def held_values_batch(self, X: np.ndarray) -> np.ndarray:
        """
        The one value held, for each row: the bristle relax rate (plant Steps 5.4, 5.5)
            d = (mu_s / mu_d) sqrt(v_s^2 + v_c^2),   mu_d = mu_k + (mu_s - mu_k) exp(-alpha_d |v_s|)
        It has a sharp corner at zero slip (v_c = 0.1 mm/s), so it is not differentiated (see the notes).
        """
        p = self.p
        v_local = self.local_velocity_batch(X)
        v_s = v_local @ self.H_drive.T - p['r'] * X[:, self.WHEELS]
        mu_d = p['mu_k'] + (p['mu_s'] - p['mu_k']) * np.exp(-p['alpha_d'] * np.abs(v_s))
        bristle_decay = (p['mu_s'] / mu_d) * np.sqrt(v_s**2 + p['lugre_creep']**2)
        return bristle_decay

    def held_values(self, x: np.ndarray) -> np.ndarray:
        """The bristle relax rate at one point."""
        bristle_decay = self.held_values_batch(x[None])[0]
        return bristle_decay

    # =======================================================================
    # The model, dx/dt = f(x, u)
    # =======================================================================

    def f_batch(self, X: np.ndarray, U: np.ndarray, bristle_decay: np.ndarray, F_n: np.ndarray = None) -> np.ndarray:
        """
        dx/dt for each row of X, with the voltages U and relax rates bristle_decay of that row held, (N, 18).
        The loads are worked out from the state (loads_batch). F_n can be passed instead, for the checks.
        """
        p = self.p
        R_theta = self.rotation_batch(X[:, 2])
        X_dot_g = X[:, self.VELOCITY]
        I = X[:, self.CURRENTS]
        omega = X[:, self.WHEELS]
        z = X[:, self.BRISTLES]

        # Velocity along the robot's axes (for the slips)
        v_local = np.einsum('nji,nj->ni', R_theta, X_dot_g)

        # Step 5.2: wheel loads, with the load transfer of this state's acceleration
        if F_n is None:
            F_n = self.loads_batch(X)

        # Step 5.6: floor force on each wheel, along its drive direction
        F_floor = -p['mu_s'] * F_n * z

        # Step 5.8: roller (axial) friction, zero when switched off
        F_a = self.roller_factor_batch(v_local) * F_n

        # Step 5.9: sum of forces in the local frame, rotated into the global frame, Newton's law there
        F_local = F_floor @ self.H_drive + F_a @ self.H_axial              # H_drive^T F, row by row
        F_global = np.einsum('nij,nj->ni', R_theta, F_local)
        X_ddot_g = F_global @ self.M_inv.T

        # Pose: d[X, Y, theta]/dt = X_dot_g
        pose_dot = X_dot_g

        # Step 5.1: motor circuit
        I_dot = (U - p['R'] * I - p['Kb'] * omega) / p['Lm']

        # Steps 5.3, 5.7: wheel torque balance, with the smoothed rolling drag
        T_roll = p['Crr'] * F_n * p['r'] * np.tanh(omega / p['roll_smoothing'])
        omega_dot = (p['Kt'] * I - T_roll - p['r'] * F_floor) / p['Jw']

        # Step 5.5: bristles, driven by the drive slip
        v_s = v_local @ self.H_drive.T - p['r'] * omega
        z_dot = (v_s - bristle_decay * z) / p['lugre_deflection']

        X_dot = np.hstack((pose_dot, X_ddot_g, I_dot, omega_dot, z_dot))
        return X_dot

    def f(self, x: np.ndarray, u: np.ndarray, bristle_decay: np.ndarray, F_n: np.ndarray = None) -> np.ndarray:
        """dx/dt at one point (the checks use this)."""
        x_dot = self.f_batch(x[None], u[None], bristle_decay[None], None if F_n is None else F_n[None])[0]
        return x_dot

    # =======================================================================
    # Step 2: linearizing
    # =======================================================================

    def jacobians_batch(self, X: np.ndarray, U: np.ndarray, bristle_decay: np.ndarray, F_n: np.ndarray = None) -> tuple:
        """
        Step 2 for each row: A = df/dx (N, 18, 18) and B = df/du (N, 18, 4), by central differences.
        For each point, each state (and each voltage) in turn is nudged up and down by a small h:
            A[:, i] = (f(x + h e_i) - f(x - h e_i)) / (2 h)
        All the nudged points of all the rows go through f_batch in ONE call (1 + 2 (18 + 4) = 45 points per row).
        The relax rates stay held; everything else is differentiated, including the loads (unless F_n is passed).
        """
        n = self.n_x
        m = self.n_u
        N = len(X)
        k = 1 + 2 * (n + m)                         # the point itself, then +h and -h in each state and each voltage
        h_x = 1e-6 * (1.0 + np.abs(X))
        h_u = 1e-6 * (1.0 + np.abs(U))

        X_all = np.repeat(X[:, None, :], k, axis=1)
        U_all = np.repeat(U[:, None, :], k, axis=1)
        i = np.arange(n)
        j = np.arange(m)
        X_all[:, 1 + i, i] += h_x                   # point 1 + i: state i nudged up
        X_all[:, 1 + n + i, i] -= h_x               # point 1 + n + i: state i nudged down
        U_all[:, 1 + 2 * n + j, j] += h_u           # then each voltage up
        U_all[:, 1 + 2 * n + m + j, j] -= h_u       # and down

        decay_all = np.repeat(bristle_decay[:, None, :], k, axis=1).reshape(N * k, 4)
        loads_all = None if F_n is None else np.repeat(F_n[:, None, :], k, axis=1).reshape(N * k, 4)
        f_all = self.f_batch(X_all.reshape(N * k, n), U_all.reshape(N * k, m), decay_all, loads_all).reshape(N, k, n)

        A = ((f_all[:, 1:1 + n] - f_all[:, 1 + n:1 + 2 * n]) / (2.0 * h_x[:, :, None])).transpose(0, 2, 1)
        B = ((f_all[:, 1 + 2 * n:1 + 2 * n + m] - f_all[:, 1 + 2 * n + m:]) / (2.0 * h_u[:, :, None])).transpose(0, 2, 1)
        return A, B

    def jacobians(self, x: np.ndarray, u: np.ndarray, bristle_decay: np.ndarray, F_n: np.ndarray = None) -> tuple:
        """A and B at one point (the checks use this)."""
        A, B = self.jacobians_batch(x[None], u[None], bristle_decay[None], None if F_n is None else F_n[None])
        return A[0], B[0]

    # =======================================================================
    # Step 3: the 20 ms model
    # =======================================================================

    def substep_maps_batch(self, X_bar: np.ndarray, U_bar: np.ndarray, h: float) -> tuple:
        """
        Step 3, for each row: one backward-Euler sub-step of length h, linearized at (x_bar, u_bar):
            x_next = x + h f(x_next)                                        (backward Euler, as the plant steps)
            f(x) ~ f_bar + A (x - x_bar) + B (u - u_bar)                     (Step 2)
            => x_next = a x + b u + c,   a = (I - h A)^-1,   b = h a B,   c = h a (f_bar - A x_bar - B u_bar)
        Returns (a, b, c): (N, 18, 18), (N, 18, 4), (N, 18).
        """
        bristle_decay = self.held_values_batch(X_bar)
        A, B = self.jacobians_batch(X_bar, U_bar, bristle_decay)
        f_bar = self.f_batch(X_bar, U_bar, bristle_decay)

        a = np.linalg.inv(np.eye(self.n_x)[None] - h * A)
        b = h * a @ B
        offset = f_bar - np.einsum('nij,nj->ni', A, X_bar) - np.einsum('nij,nj->ni', B, U_bar)
        c = h * np.einsum('nij,nj->ni', a, offset)
        return a, b, c

    def substep_map(self, x_bar: np.ndarray, u_bar: np.ndarray, h: float) -> tuple:
        """One sub-step map at one point."""
        a, b, c = self.substep_maps_batch(x_bar[None], u_bar[None], h)
        return a[0], b[0], c[0]

    def step_model(self, x0: np.ndarray, u: np.ndarray, dt: float, substeps: int) -> tuple:
        """
        Step 3, one point after another: the model over one controller step dt as x_next = A_d x0 + B_d u + c_d.
        Each of the `substeps` sub-steps is linearized at the state the previous one reached, and the maps are chained:
            A_d = a_S ... a_1,   B_d and c_d collected the same way.
        Used by the checks (and to get the first plan's points); the MPC uses linearize_along.
        Returns (A_d, B_d, c_d, x_end).
        """
        n = self.n_x
        h = dt / substeps
        A_d = np.eye(n)
        B_d = np.zeros((n, self.n_u))
        c_d = np.zeros(n)
        x = x0.copy()

        for q in range(substeps):
            a, b, c = self.substep_map(x, u, h)
            x = a @ x + b @ u + c                  # the next sub-step's point
            A_d = a @ A_d                          # chain rule: x_q+1 = a (A_d x0 + B_d u + c_d) + b u + c
            B_d = a @ B_d + b
            c_d = a @ c_d + c

        return A_d, B_d, c_d, x

    def linearize_along(self, X_sub: np.ndarray, U: np.ndarray, dt: float) -> tuple:
        """
        Step 3 for the MPC, in one batch (JP3's way): linearize at every sub-step point of a plan at once.

        X_sub: (K, S, 18), the points: S sub-steps in each of K controller steps (the MPC takes them from its last
               plan); U: (K, 4), the voltages of each controller step.
        All K * S sub-step maps come from ONE call to substep_maps_batch, then the S maps of each step are chained:
            A_d = a_S ... a_1,   B_d = a_S (... (a_1 0 + b_1) ...) + b_S,   c_d the same way.
        Returns (A_d, B_d, c_d) per step, (K, 18, 18), (K, 18, 4), (K, 18), and the sub-step maps (a, b, c), each
        (K, S, ...), which the MPC moves one step on to get the next step's points.
        """
        K, S, n = X_sub.shape
        m = self.n_u
        a, b, c = self.substep_maps_batch(X_sub.reshape(K * S, n), np.repeat(U, S, axis=0), dt / S)
        a = a.reshape(K, S, n, n)
        b = b.reshape(K, S, n, m)
        c = c.reshape(K, S, n)

        A_d = np.broadcast_to(np.eye(n), (K, n, n)).copy()
        B_d = np.zeros((K, n, m))
        c_d = np.zeros((K, n))
        for q in range(S):                         # chain rule over the sub-steps, all K steps at once
            A_d = a[:, q] @ A_d
            B_d = a[:, q] @ B_d + b[:, q]
            c_d = np.einsum('kij,kj->ki', a[:, q], c_d) + c[:, q]
        return A_d, B_d, c_d, (a, b, c)

    def from_plant(self, state) -> np.ndarray:
        """The model's state from a plant state (for checks, and the ideal observer)."""
        x = np.concatenate((state.pos_global, state.v_global, state.I, state.omega, state.z))
        return x


In [ ]:
import numpy as np
import cvxpy as cp


class MPCControllerLinearized(MPCControllerSimple):
    """
    Dana's MPCControllerSimple with the 18-state linearized model (OmniRobotLinearizedModel).
    Her class is unchanged; her cost, weights, limits and solver settings are used as they are.

    What is different (Step 4 in the model's notes):
      - the model: 18 states [pose, global velocity, currents, wheel speeds, bristles], relinearized every 5 ms
      - the offset c_j in each step's model: X_j+1 = A_j X_j + B_j U_j + c_j
      - linearized along the last plan (voltages moved one step on), all sub-step points in one batch (JP3's way)
      - the starting state: measured pose, wheel speeds, currents; velocity from her PoseVelocityEstimator;
        bristles from each wheel's torque balance (or the plant's true state, with use_true_state=True)

    Subfunctions:
      1. __init__: the linearized model first, then everything else as Dana's MPC (same arguments)
      2. initializeOptimizationProblem: Dana's, with the c_j term (lines marked ADDED)
      3. reset: Dana's, plus this class's memory
      4. observer_update: every plant step, the readings to the 1 ms observer (with use_observer=True)
      4b. estimate_bristles: z from each wheel's torque balance (without the observer)
      5. compute_optimal_u0: the state, the linearization along the plan, the solve
    """
    def __init__(self, simulation_params: dict, substeps: int = 4, use_true_state: bool = False, use_observer: bool = False,
                 **mpc_options):
        self.lin_model = OmniRobotLinearizedModel(simulation_params)
        self.substeps = int(substeps)              # backward-Euler sub-steps per controller step (4: every 5 ms)
        self.use_true_state = use_true_state       # True: the runner hands over the plant's true state (ideal observer)
        self.true_state = None
        self.use_observer = use_observer           # True: start from the 1 ms observer's estimate (its own core)
        self.observer = RobotObserver1ms(simulation_params) if use_observer else None
        super().__init__(simulation_params, **mpc_options)

    def initializeOptimizationProblem(self):
        self.n_x = self.lin_model.n_x   # ADDED: the linearized model's 18 states (Dana's model has 10)
        """
        Subfunction 2: Compiles the symbolic CVXPY optimization problem ONCE using cp.Parameter.
        Enables in-place parameter updates and warm-started OSQP execution at 50 Hz.
        Enforces box constraints on both actuator voltages and motor armature currents.
        """
        K = self.K
        n_x = self.n_x            # ADDED: 18 here (set in __init__ before this is called)
        n_u = self.n_u
        
        # Optimization Decision Variables (allocated ONCE)
        self.X_var = cp.Variable((n_x, K + 1))
        self.U_var = cp.Variable((n_u, K))
        
        # Optimization Parameters (updated in-place at each 50 Hz cycle)
        self.x0_param = cp.Parameter((n_x,))
        self.A_params = [cp.Parameter((n_x, n_x)) for _ in range(K)]
        self.B_params = [cp.Parameter((n_x, n_u)) for _ in range(K)]
        self.pos_ref_param = cp.Parameter((K + 1, 3))
        self.vel_ref_param = cp.Parameter((K + 1, 3))
        self.u_prev_param = cp.Parameter((n_u,))
        self.c_params = [cp.Parameter((n_x,)) for _ in range(K)]   # ADDED: c_j, the model's offset at each step
        
        cost = 0
        constraints = [self.X_var[:, 0] == self.x0_param]
        
        for j in range(K):
            # Tracking error stage penalties via cp.sum_squares (strict DPP compliance)
            e_pos_j = self.X_var[:3, j] - self.pos_ref_param[j]
            e_vel_j = self.X_var[3:6, j] - self.vel_ref_param[j]
            cost += cp.sum_squares(self.Q_pos_sqrt @ e_pos_j)
            cost += cp.sum_squares(self.Q_vel_sqrt @ e_vel_j)
            cost += cp.sum_squares(self.R_sqrt @ self.U_var[:, j])
            
            # Slew rate penalty and traction protection bounds
            if j == 0:
                du = self.U_var[:, j] - self.u_prev_param
            else:
                du = self.U_var[:, j] - self.U_var[:, j - 1]
            cost += cp.sum_squares(self.R_slew_sqrt @ du)
            constraints += [
                du >= -self.delta_v_max,
                du <=  self.delta_v_max
            ]
                
            # Step-varying discrete linear dynamics constraint (from OmniRobotNominalModel)
            constraints += [
                self.X_var[:, j + 1] == self.A_params[j] @ self.X_var[:, j] + self.B_params[j] @ self.U_var[:, j] + self.c_params[j]   # ADDED: + c_j
            ]
            
            # Actuator terminal voltage saturation box limits: ±12V
            constraints += [
                self.U_var[:, j] >= -self.voltage_limit,
                self.U_var[:, j] <=  self.voltage_limit
            ]
            
            # Motor driver over-current protection limits: ±20A
            constraints += [
                self.X_var[6:10, j] >= -self.current_limit,
                self.X_var[6:10, j] <=  self.current_limit
            ]
            
        # Soft terminal tracking costs via cp.sum_squares
        e_pos_K = self.X_var[:3, K] - self.pos_ref_param[K]
        e_vel_K = self.X_var[3:6, K] - self.vel_ref_param[K]
        cost += cp.sum_squares(self.P_pos_sqrt @ e_pos_K)
        cost += cp.sum_squares(self.P_vel_sqrt @ e_vel_K)
        
        # Terminal current bound
        constraints += [
            self.X_var[6:10, K] >= -self.current_limit,
            self.X_var[6:10, K] <=  self.current_limit
        ]
        
        # Pre-compile the problem
        self.problem = cp.Problem(cp.Minimize(cost), constraints)

    def reset(self):
        super().reset()
        self.omega_prev = None                     # wheel speeds at the last controller step (for the bristle estimate)
        self.U_plan = None                         # the last plan's voltages, for the next linearization
        self.sub_maps = None                       # the last plan's sub-step maps, to place the next step's points

    def observer_update(self, V: np.ndarray, omega: np.ndarray, I: np.ndarray, pose: np.ndarray = None):
        """Every plant step (1 ms): the readings go to the observer (pose only when a camera frame arrives)."""
        self.observer.update(V, omega, I, pose)

    def estimate_bristles(self, x: np.ndarray) -> np.ndarray:
        """
        The bristle state z from each wheel's torque balance, with the measured current and wheel speed:
            J_w domega/dt = K_t I - T_roll - r F   =>   F = (K_t I - T_roll - J_w domega/dt) / r,   z = -F / (mu_s F_n)
        domega/dt is the change in wheel speed since the last controller step. Before the first step, z = 0.
        """
        p = self.p
        model = self.lin_model
        omega = x[model.WHEELS]
        I = x[model.CURRENTS]
        if self.omega_prev is None:
            z = np.zeros(4)
        else:
            T_roll = p['Crr'] * model.F_n_static * p['r'] * np.tanh(omega / p['roll_smoothing'])   # small: static loads
            omega_dot = (omega - self.omega_prev) / self.dt
            F = (p['Kt'] * I - T_roll - p['Jw'] * omega_dot) / p['r']
            # z needs the loads, and the loads need z (load transfer): first guess with the static loads, then the loads
            # from that guess (model.loads), then z again
            x_guess = x.copy()
            x_guess[model.BRISTLES] = np.clip(-F / (p['mu_s'] * model.F_n_static), -1.0, 1.0)
            F_n = model.loads(x_guess)
            z = -F / (p['mu_s'] * np.maximum(F_n, 1e-3))       # a lifted wheel (F_n = 0) has no friction state
            z = np.clip(z, -1.0, 1.0)                           # |z| <= 1: the bristles hold at most mu_s F_n
        return z

    def compute_optimal_u0(self, measured_State: OmniRobotObservableState,
                           path_ref_horizon: np.ndarray, vel_ref_horizon: np.ndarray,
                           return_info: bool = False):
        """
        One controller step: the starting state, the model along the last plan, then Dana's QP.
        path_ref_horizon, vel_ref_horizon: as in Dana's MPC (vel_ref_horizon is global).
        """
        model = self.lin_model
        K = self.K

        # 1. The starting state x0 (18)
        measured_pose = np.asarray(measured_State.pos_global, dtype=float)
        v_est_pose = self.pose_estimator.estimate(measured_State.pos_global)      # global velocity (Dana's estimator)
        ref_theta0 = path_ref_horizon[0, 2]
        unwrapped_theta = (measured_pose[2] - ref_theta0 + np.pi) % (2.0 * np.pi) - np.pi + ref_theta0

        if self.use_true_state and self.true_state is not None:
            x0 = model.from_plant(self.true_state)                                # ideal observer: the plant's state
        elif self.use_observer:
            x0 = self.observer.x_hat.copy()                                       # the 1 ms observer's estimate
        else:
            x0 = np.concatenate((measured_pose, v_est_pose, measured_State.I, measured_State.omega, np.zeros(4)))
            x0[model.BRISTLES] = self.estimate_bristles(x0)
        x0[2] = unwrapped_theta
        self.omega_prev = np.asarray(measured_State.omega, dtype=float).copy()

        # 2. The model along the last plan, all sub-step points in one batch (JP3's way)
        S = self.substeps
        n = model.n_x
        if self.U_plan is None or self.sub_maps is None:
            # First step (or after a failed solve): hold the last voltages; the points from a quick rollout with
            # one sub-step per controller step, each step's point used for all its sub-steps
            U_guess = np.tile(self.u_prev, (K, 1))
            X_sub = np.zeros((K, S, n))
            x = x0.copy()
            for j in range(K):
                X_sub[j, :] = x
                _, _, _, x = model.step_model(x, U_guess[j], self.dt, 1)
        else:
            # The last plan, moved one step on: its voltages, and its sub-step maps carry the new state x0 along
            # them to give this step's points
            U_guess = np.vstack((self.U_plan[1:], self.U_plan[-1:]))
            a, b, c = (np.concatenate((maps[1:], maps[-1:])) for maps in self.sub_maps)
            X_sub = np.zeros((K, S, n))
            x = x0.copy()
            for j in range(K):
                for q in range(S):
                    X_sub[j, q] = x
                    x = a[j, q] @ x + b[j, q] @ U_guess[j] + c[j, q]
        A_d, B_d, c_d, self.sub_maps = model.linearize_along(X_sub, U_guess, self.dt)
        for j in range(K):
            self.A_params[j].value = A_d[j]
            self.B_params[j].value = B_d[j]
            self.c_params[j].value = c_d[j]

        # 3. Dana's QP, as in her MPC
        self.x0_param.value = x0
        self.pos_ref_param.value = path_ref_horizon
        self.vel_ref_param.value = vel_ref_horizon
        self.u_prev_param.value = self.u_prev
        self.problem.solve(solver=cp.OSQP, eps_abs=1e-3, eps_rel=1e-3, max_iter=4000, warm_start=True, verbose=False)

        self.last_status = self.problem.status
        self.fallback_used = (self.problem.status not in ["optimal", "optimal_inaccurate"] or self.U_var.value is None)
        if self.fallback_used:
            u_0 = self.u_prev.copy()
            self.U_plan = None
            self.sub_maps = None
        else:
            u_0 = self.U_var.value[:, 0].copy()
            self.last_X_opt = self.X_var.value.copy()
            self.U_plan = self.U_var.value.T.copy()
        self.u_prev = u_0.copy()

        if return_info:
            return u_0, {'optimal': not self.fallback_used, 'status': self.last_status, 'fallback': self.fallback_used,
                         'x0': x0.copy()}
        return u_0


### `FastPoseVelocityEstimator` and `MPCControllerSimple1msEstimator`: a 1 ms velocity estimate (added in this copy)

Dana's MPC gets its velocity from her `PoseVelocityEstimator`, called once per 20 ms controller step: the pose difference over 20 ms, an average about 10 ms old.

`FastPoseVelocityEstimator` runs her same `PoseVelocityEstimator` with `dt` = the plant step (1 ms), updated after every plant step (on the robot it would run on its own core). It keeps the latest estimate. `MPCControllerSimple1msEstimator` is her MPC with that estimator in place of hers: her MPC calls `self.pose_estimator.estimate(...)` every 20 ms, and gets the latest 1 ms estimate. Her classes are unchanged.


In [ ]:
import numpy as np


class FastPoseVelocityEstimator:
    """
    Dana's PoseVelocityEstimator run every plant step (1 ms), keeping its latest estimate for the MPC.
    Same interface as hers (estimate, reset), so her MPC can use it as its pose_estimator.

    Subfunctions:
      1. __init__: her estimator, with dt = the plant step
      2. update: every plant step (1 ms), with the newest pose
      3. estimate: what her MPC calls every 20 ms: the latest 1 ms estimate (global velocity)
      4. reset: as hers
    """
    def __init__(self, plant_dt: float):
        self.fast_estimator = PoseVelocityEstimator(dt=plant_dt)
        self.latest = np.zeros(3)

    def update(self, current_pos_global: np.ndarray):
        """Every 1 ms: the pose difference over the last plant step (her estimator, at 1 ms)."""
        self.latest = self.fast_estimator.estimate(current_pos_global)

    def estimate(self, current_pos_global: np.ndarray) -> np.ndarray:
        """Called by Dana's MPC every 20 ms: the latest 1 ms estimate [vx_g, vy_g, omega_yaw]."""
        return self.latest.copy()

    def reset(self, initial_pos: np.ndarray = None):
        self.fast_estimator.reset(initial_pos)
        self.latest = np.zeros(3)


class MPCControllerSimple1msEstimator(MPCControllerSimple):
    """
    Dana's MPCControllerSimple with its velocity from a 1 ms estimator (FastPoseVelocityEstimator) instead of her
    20 ms pose difference. Everything else is hers, unchanged.

    Subfunctions:
      1. __init__: her MPC (same arguments), then the 1 ms estimator in place of her pose_estimator
      2. estimator_update: called by the runner after every plant step (1 ms)
    """
    def __init__(self, simulation_params: dict, **mpc_options):
        super().__init__(simulation_params, **mpc_options)
        self.pose_estimator = FastPoseVelocityEstimator(simulation_params['dt'])

    def estimator_update(self, measured_State: OmniRobotObservableState):
        """Every plant step (1 ms): the newest pose goes to the 1 ms estimator."""
        self.pose_estimator.update(np.asarray(measured_State.pos_global, dtype=float))


### `RobotObserver1ms`: the 1 ms observer (added in this copy)

Runs every plant step (1 ms), on its own core on the robot. Its estimate $\hat{\mathbf{x}}$ has the same 18 states as the linearized model, so the linearized MPC can start from it.

What it can measure (no IMU; the camera on time):
* **every 1 ms:** the wheel speeds $\omega$ (encoders) and the motor currents $i$ (drivers);
* **every 20 ms:** the pose $[X, Y, \theta]$ (camera).

Every 1 ms:
1. **Predict:** one 1 ms backward-Euler step of the linearized model (`substep_map`) with the voltages just applied. Between camera frames this is what carries the pose and velocity forward: the velocity changes with the friction forces, which come from the bristle state $z$.
2. **Correct with the 1 ms sensors:**
   * the currents and wheel speeds are measured, so the estimate takes the measured values;
   * the bristle state: each wheel's torque balance with the measured current and its acceleration over this 1 ms gives the floor force, $F = (K_t i - T_{roll} - J_w\dot\omega)/r$, so a measured $z_m = -F/(\mu_s F_n)$. The estimate moves toward it: $\hat z \leftarrow \hat z + L_z(z_m - \hat z)$.

Every 20 ms, when the camera pose arrives:

3. **Correct with the camera:** with the pose error $\mathbf{e} = \mathbf{p}_{camera} - \hat{\mathbf{p}}$, the pose moves toward the camera's, $\hat{\mathbf{p}} \leftarrow \hat{\mathbf{p}} + L_p\,\mathbf{e}$, and the velocity is corrected by how much the pose drifted over the last 20 ms, $\hat{\dot{\mathbf{X}}} \leftarrow \hat{\dot{\mathbf{X}}} + L_v\,\mathbf{e} / T_{camera}$ (an alpha-beta correction).

The gains $L_z$, $L_p$, $L_v$ are fixed numbers for now (a Kalman gain would work them out from the noise levels; later).


In [ ]:
import numpy as np


class RobotObserver1ms:
    """
    1 ms observer: predicts with the linearized model, corrects with the 1 ms sensors (wheel speeds, currents)
    and with the camera pose when it arrives (every 20 ms). Estimate x_hat: the linearized model's 18 states.

    Subfunctions:
      1. __init__: the model and the gains
      2. reset: start from a known state (the robot at rest at its start pose)
      3. predict: one plant step of the model, with the voltages applied
      4. correct_fast: the measured wheel speeds and currents, and the bristle state from the torque balance
      5. correct_camera: the camera pose (and the velocity from how far the pose drifted)
      6. update: what the runner calls every plant step
    """
    def __init__(self, simulation_params: dict, gain_bristle: float = 1.0, gain_pose: float = 1.0,
                 gain_velocity: float = 0.5, camera_period: float = 0.02):
        self.p = simulation_params
        self.model = OmniRobotLinearizedModel(simulation_params)
        self.dt = simulation_params['dt']          # the observer runs every plant step (1 ms)
        self.gain_bristle = gain_bristle           # L_z: how far z moves toward the torque-balance value each 1 ms
        self.gain_pose = gain_pose                 # L_p: how far the pose moves toward the camera's
        self.gain_velocity = gain_velocity         # L_v: how much of the pose drift goes into the velocity
        self.camera_period = camera_period         # time between camera frames (s)
        self.reset()

    def reset(self, x0: np.ndarray = None):
        """Start from a known state (default: at rest at the origin)."""
        self.x_hat = np.zeros(self.model.n_x) if x0 is None else x0.copy()
        self.omega_prev = None                     # the last measured wheel speeds, for the wheel acceleration

    def predict(self, V: np.ndarray):
        """One plant step (1 ms) of the linearized model from the estimate, with the voltages applied."""
        a, b, c = self.model.substep_map(self.x_hat, V, self.dt)
        self.x_hat = a @ self.x_hat + b @ V + c

    def correct_fast(self, omega: np.ndarray, I: np.ndarray):
        """
        The 1 ms sensors: the measured currents and wheel speeds replace the estimate's, and the bristle state moves
        toward the value from each wheel's torque balance:
            J_w domega/dt = K_t I - T_roll - r F   =>   F = (K_t I - T_roll - J_w domega/dt) / r,   z_m = -F / (mu_s F_n)
        """
        p = self.p
        model = self.model
        self.x_hat[model.CURRENTS] = I
        self.x_hat[model.WHEELS] = omega

        if self.omega_prev is not None:
            omega_dot = (omega - self.omega_prev) / self.dt                  # wheel acceleration over this 1 ms
            F_n = model.loads(self.x_hat)
            T_roll = p['Crr'] * F_n * p['r'] * np.tanh(omega / p['roll_smoothing'])
            F = (p['Kt'] * I - T_roll - p['Jw'] * omega_dot) / p['r']       # floor force on each wheel
            z_measured = -F / (p['mu_s'] * np.maximum(F_n, 1e-3))           # a lifted wheel has no friction state
            z_measured = np.clip(z_measured, -1.0, 1.0)                     # |z| <= 1: at most mu_s F_n
            z_hat = self.x_hat[model.BRISTLES]
            self.x_hat[model.BRISTLES] = z_hat + self.gain_bristle * (z_measured - z_hat)
        self.omega_prev = omega.copy()

    def correct_camera(self, pose: np.ndarray):
        """
        The camera pose (every camera_period): the pose error e = pose - pose_hat (heading wrapped to +-pi),
            pose_hat += L_p e,    velocity_hat += L_v e / camera_period
        """
        model = self.model
        error = pose - self.x_hat[model.POSE]
        error[2] = (error[2] + np.pi) % (2.0 * np.pi) - np.pi
        self.x_hat[model.POSE] = self.x_hat[model.POSE] + self.gain_pose * error
        self.x_hat[model.VELOCITY] = self.x_hat[model.VELOCITY] + self.gain_velocity * error / self.camera_period

    def update(self, V: np.ndarray, omega: np.ndarray, I: np.ndarray, pose: np.ndarray = None):
        """
        Every plant step (1 ms): predict with the voltages applied over the step, then correct with the readings at
        its end. pose: the camera pose when a frame arrives, None between frames.
        """
        self.predict(V)
        self.correct_fast(omega, I)
        if pose is not None:
            self.correct_camera(pose)
